In [ ]:
# @title Data Importation
import pandas as pd
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)
df[["date_submitted", "county", "facility",
    "quality_care_score"
]].head(2)


## **For DnA use alone**

In [ ]:
# @title Trend Quality of Care Over Time
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display
import numpy as np  # Import numpy for nan values
from scipy import stats  # Import scipy for linear regression
import matplotlib.dates as mdates  # Import for date conversion

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure 'date_submitted' is in datetime format
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed

# Global invalid values
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Section indicators
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']

# Technical Quality After Birth (29 indicators)
# = former "Technical Quality Immediately Following Birth" (21)
#   + former "Maternal Vitals & Documentation" (8)
tqab_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                   'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                   'method_blood_loss', 'proper_vdrape_use', 'maternal_bleeding_assessed',
                   'administer_vitamin_k', 'timely_vitamink_administration',
                   'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation',
                   'take_mother_bp', 'timely_mother_bp',
                   'take_mother_temperature', 'timely_mother_temp',
                   'take_mother_pulse', 'timely_mother_pulse',
                   'apgar_min1', 'apgar_min5']

# EMOTIVE (8 indicators)
emotive_indicators = ['uterine_massage_initiated', 'timely_uterine_massage',
                      'oxytocin_first_line', 'timely_oxytocin_administration',
                      'txa_administered', 'timely_txa_administration',
                      'iv_access_fluids', 'timely_iv_establishment']

infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']
partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']
lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# Map section names to their indicators
section_mapping = {
    "Birth Preparedness": birth_preparedness,
    "Technical Quality After Birth": tqab_indicators,
    "Postbirth Infection Prevention": infection_indicators,
    "Respectful Maternity Care": rmc_indicators,
    "Avoidance Harmful Practices": harm_indicators,
    "Partograph Review": partograph_indicators,
    "Labor Care Guide Review": lcg_indicators,
    "EMOTIVE": emotive_indicators,
}

# Unified score computation function
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# Create filter widgets
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

section_widget = widgets.Dropdown(
    options=["All"] + sorted(section_mapping.keys()),
    value="All",
    description="Section:"
)

# Function to create the interactive line chart
def plot_daily_average_score(county, facility, cadre, section):
    filtered_df = df.copy()

    # Apply filters
    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]
    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]
    if cadre != "All":
        filtered_df = filtered_df[filtered_df['cadre'] == cadre]

    # Drop rows with invalid dates and sort by date
    df_cleaned = filtered_df.dropna(subset=['date_submitted']).sort_values('date_submitted')

    if df_cleaned.empty:
        print("No data available for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    series_to_plot = pd.Series(dtype=float)
    plot_title = ''

    if section == "All":
        # Group by date and calculate the mean quality_care_score for daily plots
        series_to_plot = df_cleaned.groupby(df_cleaned['date_submitted'].dt.date)['quality_care_score'].mean()
        plot_title = 'Average Quality of Care Score Over Time'
    else:
        # Calculate daily average score for the selected section
        selected_indicators = section_mapping[section]
        daily_section_scores = {}

        for single_date in sorted(df_cleaned['date_submitted'].dt.normalize().unique()):
            df_for_date = df_cleaned[df_cleaned['date_submitted'].dt.normalize() == single_date].copy()
            if df_for_date.empty:
                daily_section_scores[single_date] = np.nan
                continue

            scores_for_section_on_date = []
            for col in selected_indicators:
                if col in df_for_date.columns:
                    score = compute_score_unified(df_for_date[col], col)
                    if not pd.isna(score):
                        scores_for_section_on_date.append(score)

            if scores_for_section_on_date:
                daily_section_scores[single_date] = np.mean(scores_for_section_on_date)
            else:
                daily_section_scores[single_date] = np.nan

        series_to_plot = pd.Series(daily_section_scores)
        plot_title = f'Daily Average {section} Score Over Time'

    # Convert the index to datetime for proper plotting
    series_to_plot.index = pd.to_datetime(series_to_plot.index)
    series_to_plot = series_to_plot.dropna()  # Drop dates with no calculable score

    if series_to_plot.empty:
        print("No calculable scores for the selected filters and section.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No calculable scores for the selected filters and section.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the line chart
    plt.figure(figsize=(14, 7))
    plt.plot(series_to_plot.index, series_to_plot.values * 100, marker='o', linestyle='-', color='purple')

    # Add trend line
    x_values_numeric = mdates.date2num(series_to_plot.index)
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_values_numeric, series_to_plot.values * 100)
    trend_line = slope * x_values_numeric + intercept
    plt.plot(series_to_plot.index, trend_line, color='gold', linestyle='--', label='Trend Line')
    plt.legend()

    plt.title(plot_title, fontsize=16)
    plt.xlabel('Date Submitted', fontsize=12)
    plt.ylabel('Average Score (%)', fontsize=12)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('lightgrey')
    ax.spines['bottom'].set_color('lightgrey')

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Interactive dashboard setup
ui = widgets.VBox([county_widget, facility_widget, cadre_widget, section_widget])

out = widgets.interactive_output(
    plot_daily_average_score,
    {
        "county": county_widget,
        "facility": facility_widget,
        "cadre": cadre_widget,
        "section": section_widget
    }
)

display(ui, out)

In [ ]:
# @title Trend Quality of Care by Month
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display
import numpy as np  # Import numpy for nan values
from scipy import stats  # Import scipy for linear regression
import matplotlib.dates as mdates  # Import for date conversion

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure 'date_submitted' is in datetime format
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month'] = df['date_submitted'].dt.strftime('%Y-%m')  # Add month column for grouping

# Global invalid values
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Section indicators
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']

# Technical Quality After Birth (29 indicators)
# = former "Technical Quality Immediately Following Birth" (21)
#   + former "Maternal Vitals & Documentation" (8)
tqab_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                   'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                   'method_blood_loss', 'proper_vdrape_use', 'maternal_bleeding_assessed',
                   'administer_vitamin_k', 'timely_vitamink_administration',
                   'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation',
                   'take_mother_bp', 'timely_mother_bp',
                   'take_mother_temperature', 'timely_mother_temp',
                   'take_mother_pulse', 'timely_mother_pulse',
                   'apgar_min1', 'apgar_min5']

# EMOTIVE (8 indicators)
emotive_indicators = ['uterine_massage_initiated', 'timely_uterine_massage',
                      'oxytocin_first_line', 'timely_oxytocin_administration',
                      'txa_administered', 'timely_txa_administration',
                      'iv_access_fluids', 'timely_iv_establishment']

infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']
partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']
lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# Map section names to their indicators
section_mapping = {
    "Birth Preparedness": birth_preparedness,
    "Technical Quality After Birth": tqab_indicators,
    "Postbirth Infection Prevention": infection_indicators,
    "Respectful Maternity Care": rmc_indicators,
    "Avoidance Harmful Practices": harm_indicators,
    "Partograph Review": partograph_indicators,
    "Labor Care Guide Review": lcg_indicators,
    "EMOTIVE": emotive_indicators,
}

# Unified score computation function
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# Create filter widgets
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

section_widget = widgets.Dropdown(
    options=["All"] + sorted(section_mapping.keys()),
    value="All",
    description="Section:"
)

# Function to create the interactive line chart
def plot_daily_average_score(county, facility, cadre, section):
    filtered_df = df.copy()

    # Apply filters
    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]
    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]
    if cadre != "All":
        filtered_df = filtered_df[filtered_df['cadre'] == cadre]

    # Drop rows with invalid dates and sort by date
    df_cleaned = filtered_df.dropna(subset=['date_submitted', 'month']).sort_values('date_submitted')

    if df_cleaned.empty:
        print("No data available for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    series_to_plot = pd.Series(dtype=float)
    plot_title = ''

    if section == "All":
        # Group by month and calculate the mean quality_care_score for monthly plots
        series_to_plot = df_cleaned.groupby(df_cleaned['month'])['quality_care_score'].mean()
        plot_title = 'Average Quality of Care Score Over Time (by Month)'
    else:
        # Calculate monthly average score for the selected section
        selected_indicators = section_mapping[section]
        monthly_section_scores = {}

        for single_month in sorted(df_cleaned['month'].unique()):
            df_for_month = df_cleaned[df_cleaned['month'] == single_month].copy()
            if df_for_month.empty:
                monthly_section_scores[single_month] = np.nan
                continue

            scores_for_section_on_month = []
            for col in selected_indicators:
                if col in df_for_month.columns:
                    score = compute_score_unified(df_for_month[col], col)
                    if not pd.isna(score):
                        scores_for_section_on_month.append(score)

            if scores_for_section_on_month:
                monthly_section_scores[single_month] = np.mean(scores_for_section_on_month)
            else:
                monthly_section_scores[single_month] = np.nan

        series_to_plot = pd.Series(monthly_section_scores)
        plot_title = f'Monthly Average {section} Score Over Time'

    # Convert the index to datetime for proper plotting
    series_to_plot.index = pd.to_datetime(series_to_plot.index)
    series_to_plot = series_to_plot.dropna()  # Drop dates with no calculable score

    if series_to_plot.empty:
        print("No calculable scores for the selected filters and section.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No calculable scores for the selected filters and section.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the line chart
    plt.figure(figsize=(14, 7))
    plt.plot(series_to_plot.index, series_to_plot.values * 100, marker='o', linestyle='-', color='purple')

    plt.title(plot_title, fontsize=18)
    plt.xlabel('Month', fontsize=16)
    plt.ylabel('Average Score (%)', fontsize=16)
    plt.ylim(0, 100)  # Set y-axis scale from 0% to 100%

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('lightgrey')
    ax.spines['bottom'].set_color('lightgrey')

    # Format x-axis to show month names
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    # Use MonthLocator for clearer monthly labels without repetition
    ax.xaxis.set_major_locator(mdates.MonthLocator())

    # Tick label font sizes
    plt.xticks(rotation=45, fontsize=12)
    plt.yticks(fontsize=12)

    # Add data labels above each point, black and bold
    for x, y in zip(series_to_plot.index, series_to_plot.values * 100):
        if not pd.isna(y):
            plt.text(x, y + 2, f"{y:.1f}%", ha='center', va='bottom',
                     fontsize=11, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard setup
ui = widgets.VBox([county_widget, facility_widget, cadre_widget, section_widget])

out = widgets.interactive_output(
    plot_daily_average_score,
    {
        "county": county_widget,
        "facility": facility_widget,
        "cadre": cadre_widget,
        "section": section_widget
    }
)

display(ui, out)

In [ ]:
# @title Trend Quality of Care by Month II
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display
import numpy as np  # Import numpy for nan values
import scipy.stats as stats  # Import scipy for linear regression
import matplotlib.dates as mdates  # Import for date conversion

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure 'date_submitted' is in datetime format
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month'] = df['date_submitted'].dt.strftime('%Y-%m')  # Add month column for grouping

# Global invalid values
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Section indicators
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']

# Technical Quality Immediately Following Birth (21 indicators)
tqifb_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                    'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                    'method_blood_loss', 'proper_vdrape_use', 'maternal_bleeding_assessed',
                    'administer_vitamin_k', 'timely_vitamink_administration',
                    'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation']

# Maternal Vitals & Documentation (8 indicators)
vitals_indicators = ['take_mother_bp', 'timely_mother_bp',
                     'take_mother_temperature', 'timely_mother_temp',
                     'take_mother_pulse', 'timely_mother_pulse',
                     'apgar_min1', 'apgar_min5']

# EMOTIVE (8 indicators)
emotive_indicators = ['uterine_massage_initiated', 'timely_uterine_massage',
                      'oxytocin_first_line', 'timely_oxytocin_administration',
                      'txa_administered', 'timely_txa_administration',
                      'iv_access_fluids', 'timely_iv_establishment']

infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']
partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']
lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# Map section names to their indicators
section_mapping = {
    "Birth Preparedness": birth_preparedness,
    "Technical Quality Immediately Following Birth": tqifb_indicators,
    "Maternal Vitals & Documentation": vitals_indicators,
    "Postbirth Infection Prevention": infection_indicators,
    "Respectful Maternity Care": rmc_indicators,
    "Avoidance Harmful Practices": harm_indicators,
    "Partograph Review": partograph_indicators,
    "Labor Care Guide Review": lcg_indicators,
    "EMOTIVE": emotive_indicators,
}

# Unified score computation function
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# Create filter widgets
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

section_widget = widgets.Dropdown(
    options=["All"] + sorted(section_mapping.keys()),
    value="All",
    description="Section:"
)

# Function to create the interactive line chart
def plot_daily_average_score(county, facility, cadre, section):
    # Start with a copy of the original df
    filtered_df_base = df.copy()

    # Apply facility and cadre filters first
    if facility != "All":
        filtered_df_base = filtered_df_base[filtered_df_base['facility'] == facility]
    if cadre != "All":
        filtered_df_base = filtered_df_base[filtered_df_base['cadre'] == cadre]

    # Drop rows with invalid dates and sort by date *after* facility/cadre filtering
    df_cleaned_initial = filtered_df_base.dropna(subset=['date_submitted', 'month']).sort_values('date_submitted')

    if df_cleaned_initial.empty:
        print("No data available for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Determine which counties to iterate over for plotting
    counties_to_plot = []
    if county == "All":
        counties_to_plot = sorted(df_cleaned_initial['county'].dropna().unique().tolist())
    else:
        # Check if the selected county exists in the filtered data
        if county not in df_cleaned_initial['county'].unique():
            print(f"No data available for {county} county with the selected filters.")
            plt.figure(figsize=(14, 7))
            plt.text(0.5, 0.5, f'No data available for {county} county with the selected filters.',
                     horizontalalignment='center', verticalalignment='center',
                     fontsize=14, color='gray')
            plt.axis('off')
            plt.show()
            return
        counties_to_plot = [county]

    if not counties_to_plot:  # If no counties after filters (e.g. if 'All' was selected but no county data exists)
        print("No data available for the selected filters across any county.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters across any county.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    plt.figure(figsize=(14, 7))
    has_plotted_any_line = False
    last_series_plotted = None  # To store the series for data labels if only one line

    # Prepare plotting title
    plot_title = ''
    if section == "All":
        plot_title = 'Average Quality of Care Score Over Time (by Month)'
    else:
        plot_title = f'Monthly Average {section} Score Over Time'

    if county == "All" and len(counties_to_plot) > 1:
        plot_title += ""
    elif county != "All":
        plot_title += f" for {county} County"

    for current_county in counties_to_plot:
        df_for_current_county = df_cleaned_initial[df_cleaned_initial['county'] == current_county].copy()
        if df_for_current_county.empty:
            continue

        series_to_plot_for_county = pd.Series(dtype=float)
        plot_label = current_county  # Use county name as label for legend

        if section == "All":
            series_to_plot_for_county = df_for_current_county.groupby(df_for_current_county['month'])['quality_care_score'].mean()
        else:
            selected_indicators = section_mapping[section]
            monthly_section_scores = {}
            for single_month in sorted(df_for_current_county['month'].unique()):
                df_for_month = df_for_current_county[df_for_current_county['month'] == single_month].copy()
                if df_for_month.empty:
                    monthly_section_scores[single_month] = np.nan
                    continue

                scores_for_section_on_month = []
                for col in selected_indicators:
                    if col in df_for_month.columns:
                        score = compute_score_unified(df_for_month[col], col)
                        if not pd.isna(score):
                            scores_for_section_on_month.append(score)

                if scores_for_section_on_month:
                    monthly_section_scores[single_month] = np.mean(scores_for_section_on_month)
                else:
                    monthly_section_scores[single_month] = np.nan
            series_to_plot_for_county = pd.Series(monthly_section_scores)

        # Convert the index to datetime for proper plotting
        series_to_plot_for_county.index = pd.to_datetime(series_to_plot_for_county.index)
        series_to_plot_for_county = series_to_plot_for_county.dropna()

        if not series_to_plot_for_county.empty:
            plt.plot(series_to_plot_for_county.index, series_to_plot_for_county.values * 100, marker='o', linestyle='-', label=plot_label)
            has_plotted_any_line = True
            if len(counties_to_plot) == 1:  # Store if only one line is expected for data labels
                last_series_plotted = series_to_plot_for_county

    if not has_plotted_any_line:
        print("No calculable scores for the selected filters and section.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No calculable scores for the selected filters and section.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    plt.title(plot_title, fontsize=18)
    plt.xlabel('Month', fontsize=16)
    plt.ylabel('Average Score (%)', fontsize=16)
    plt.ylim(70, 100)  # Set y-axis scale from 70% to 100%
    plt.yticks(np.arange(70, 101, 5))  # Ticks every 5%

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('lightgrey')
    ax.spines['bottom'].set_color('lightgrey')

    # Format x-axis to show month names
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    # Use MonthLocator for clearer monthly labels without repetition
    ax.xaxis.set_major_locator(mdates.MonthLocator())

    # Tick label font sizes
    plt.xticks(rotation=45, fontsize=12)
    plt.yticks(np.arange(70, 101, 5), fontsize=12)

    # Add legend if multiple lines are plotted
    if county == "All" and len(counties_to_plot) > 1:
        plt.legend(title='County', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Add data labels ONLY if plotting a single line (i.e., a specific county is selected)
    if len(counties_to_plot) == 1 and last_series_plotted is not None:
        for x, y in zip(last_series_plotted.index, last_series_plotted.values * 100):
            if not pd.isna(y):
                plt.text(x, y + 2, f"{y:.1f}%", ha='center', va='bottom',
                         fontsize=11, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard setup
ui = widgets.VBox([county_widget, facility_widget, cadre_widget, section_widget])

out = widgets.interactive_output(
    plot_daily_average_score,
    {
        "county": county_widget,
        "facility": facility_widget,
        "cadre": cadre_widget,
        "section": section_widget
    }
)

display(ui, out)

In [ ]:
# @title Average Section Scores Over Time

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure 'date_submitted' is in datetime format
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted']) # Drop rows where date couldn't be parsed

# Global invalid values
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Section indicators (from 6vr_qlneDJH-)
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']
tqab_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                   'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                   'method_blood_loss', 'proper_vdrape_use', 'uterine_massage_initiated', 'oxytocin_first_line', 'txa_administered', 'iv_access_fluids', 'take_mother_bp', 'timely_mother_bp',
                   'take_mother_temperature', 'timely_mother_temp', 'take_mother_pulse', 'timely_mother_pulse', 'apgar_min1', 'apgar_min5', 'administer_vitamin_k', 'timely_vitamink_administration',
                   'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation', 'maternal_bleeding_assessed']
infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']
partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']
lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# Map section names to their indicators
section_mapping = {
    "Birth Preparedness": birth_preparedness,
    "Technical Quality Postbirth": tqab_indicators,
    "Postbirth Infection Prevention": infection_indicators,
    "Respectful Maternity Care": rmc_indicators,
    "Avoidance Harmful Practices": harm_indicators,
    "Partograph Review": partograph_indicators,
    "Labor Care Guide Review": lcg_indicators,
}

# Unified score computation function (from 6vr_qlneDJH-)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# Create filter widgets
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

# Function to calculate and plot section scores over time
def plot_section_scores_over_time(county, facility, cadre):
    filtered_df = df.copy()

    # Apply filters
    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]
    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]
    if cadre != "All":
        filtered_df = filtered_df[filtered_df['cadre'] == cadre]

    df_cleaned = filtered_df.dropna(subset=['date_submitted']).sort_values('date_submitted')

    if df_cleaned.empty:
        print("No data available for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Prepare a DataFrame to store daily section scores
    daily_section_scores = pd.DataFrame(index=sorted(df_cleaned['date_submitted'].dt.normalize().unique()))
    daily_section_scores.index.name = 'date_submitted'

    for section_name, indicators in section_mapping.items():
        section_scores_daily_list = []
        for single_date in daily_section_scores.index:
            df_for_date = df_cleaned[df_cleaned['date_submitted'].dt.normalize() == single_date]
            if df_for_date.empty:
                section_scores_daily_list.append(np.nan)
                continue

            scores_for_section_on_date = []
            for col in indicators:
                if col in df_for_date.columns:
                    score = compute_score_unified(df_for_date[col], col)
                    if not pd.isna(score):
                        scores_for_section_on_date.append(score)
            if scores_for_section_on_date:
                section_scores_daily_list.append(np.mean(scores_for_section_on_date))
            else:
                section_scores_daily_list.append(np.nan)
        daily_section_scores[section_name] = section_scores_daily_list

    # Convert scores to percentage
    daily_section_scores = daily_section_scores * 100

    # Drop any rows (dates) where all section scores are NaN
    daily_section_scores = daily_section_scores.dropna(how='all')

    if daily_section_scores.empty:
        print("No calculable section scores over time for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No calculable section scores over time for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return


    # Plotting the multiple line chart
    plt.figure(figsize=(16, 8))
    for column in daily_section_scores.columns:
        plt.plot(daily_section_scores.index, daily_section_scores[column], marker='o', linestyle='-', label=column)

    plt.title('Average Section Scores Over Time', fontsize=16)
    plt.xlabel('Date Submitted', fontsize=12)
    plt.ylabel('Average Score (%)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Place legend outside the plot
    plt.grid(False) # Removed grid lines

    # Customize x-axis date format
    date_format = '%y-%m-%d'
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter(date_format))

    # Customize borders
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('lightgrey')
    ax.spines['bottom'].set_visible(False) # Removed bottom border

    plt.tight_layout()
    plt.show()

# Interactive dashboard setup
ui = widgets.VBox([county_widget, facility_widget, cadre_widget])

out = widgets.interactive_output(
    plot_section_scores_over_time,
    {
        "county": county_widget,
        "facility": facility_widget,
        "cadre": cadre_widget
    }
)

display(ui, out)

In [ ]:
# @title Average Section Scores Over Time (Heatmap)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap # Import LinearSegmentedColormap

# Data Importation (for self-containment, ensuring df is available)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure 'date_submitted' is in datetime format
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted']) # Drop rows where date couldn't be parsed

# Global invalid values (copied from previous cell for self-containment)
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Section indicators (copied from previous cell for self-containment)
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']
tqab_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                   'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                   'method_blood_loss', 'proper_vdrape_use', 'uterine_massage_initiated', 'oxytocin_first_line', 'txa_administered', 'iv_access_fluids', 'take_mother_bp', 'timely_mother_bp',
                   'take_mother_temperature', 'timely_mother_temp', 'take_mother_pulse', 'timely_mother_pulse', 'apgar_min1', 'apgar_min5', 'administer_vitamin_k', 'timely_vitamink_administration',
                   'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation', 'maternal_bleeding_assessed']
infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']
partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']
lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# Map section names to their indicators (copied from previous cell for self-containment)
section_mapping = {
    "Birth Preparedness": birth_preparedness,
    "Technical Quality Postbirth": tqab_indicators,
    "Postbirth Infection Prevention": infection_indicators,
    "Respectful Maternity Care": rmc_indicators,
    "Avoidance Harmful Practices": harm_indicators,
    "Partograph Review": partograph_indicators,
    "Labor Care Guide Review": lcg_indicators,
}

# Unified score computation function (copied from previous cell for self-containment)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# Create filter widgets (same as in previous cell to maintain state)
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

# Define the custom color map with sharp transitions for discrete ranges
# Brown for < 50, Yellow for 50-84, Green for >= 85
colors_for_cmap =[
    (0.0, "#B2182B"),   # Deep red: very low scores (0)
    (0.40, "#D6604D"),  # Soft red-orange (~40)
    (0.60, "#F4A582"),  # Orange transition (~60)
    (0.75, "#FEE08B"),  # Yellow (~75)
    (0.85, "#FFFFBF"),  # Light yellow near threshold (~85)
    (1.0, "#1A9850")    # Green: high scores (100)
]
cmap_custom = LinearSegmentedColormap.from_list("custom_diverging", colors_for_cmap)

# Function to calculate and plot section scores heatmap
def plot_section_scores_heatmap(county, facility, cadre):
    filtered_df = df.copy()

    # Apply filters
    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]
    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]
    if cadre != "All":
        filtered_df = filtered_df[filtered_df['cadre'] == cadre]

    df_cleaned = filtered_df.dropna(subset=['date_submitted']).sort_values('date_submitted')

    if df_cleaned.empty:
        print("No data available for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Prepare a DataFrame to store daily section scores
    daily_section_scores = pd.DataFrame(index=sorted(df_cleaned['date_submitted'].dt.normalize().unique()))
    daily_section_scores.index.name = 'date_submitted'

    for section_name, indicators in section_mapping.items():
        section_scores_daily_list = []
        for single_date in daily_section_scores.index:
            df_for_date = df_cleaned[df_cleaned['date_submitted'].dt.normalize() == single_date]
            if df_for_date.empty:
                section_scores_daily_list.append(np.nan)
                continue

            scores_for_section_on_date = []
            for col in indicators:
                if col in df_for_date.columns:
                    score = compute_score_unified(df_for_date[col], col)
                    if not pd.isna(score):
                        scores_for_section_on_date.append(score)
            if scores_for_section_on_date:
                section_scores_daily_list.append(np.mean(scores_for_section_on_date))
            else:
                section_scores_daily_list.append(np.nan)
        daily_section_scores[section_name] = section_scores_daily_list

    # Convert scores to percentage
    daily_section_scores = daily_section_scores * 100

    # Drop any rows (dates) where all section scores are NaN
    daily_section_scores = daily_section_scores.dropna(how='all')

    if daily_section_scores.empty: # Check again after dropping NaNs
        print("No calculable section scores over time for the selected filters.")
        plt.figure(figsize=(14, 7))
        plt.text(0.5, 0.5, 'No calculable section scores over time for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Reshape for heatmap: dates as index, sections as columns
    heatmap_data = daily_section_scores.copy()
    heatmap_data.index = heatmap_data.index.strftime('%y-%m-%d') # Format dates for readability

    # Plotting the heatmap
    plt.figure(figsize=(16, min(len(heatmap_data.columns) * 0.8 + 2, 20))) # Dynamic height
    sns.heatmap(
        heatmap_data.T, # Transpose to have sections on Y-axis and dates on X-axis
        annot=False,     # Show the score values on the heatmap
        fmt=".1f",      # Format annotations to one decimal place
        cmap=cmap_custom, # Use the custom color map
        linewidths=.5,  # Add lines between cells
        cbar_kws={'label': 'Average Score (%)'},
        vmin=0,         # Set vmin for consistent color mapping
        vmax=100        # Set vmax for consistent color mapping
    )

    plt.title('Average Section Scores Over Time (Heatmap)', fontsize=16)
    plt.xlabel('Date Submitted', fontsize=12)
    plt.ylabel('Section', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Interactive dashboard setup for heatmap
ui_heatmap = widgets.VBox([county_widget, facility_widget, cadre_widget])

out_heatmap = widgets.interactive_output(
    plot_section_scores_heatmap,
    {
        "county": county_widget,
        "facility": facility_widget,
        "cadre": cadre_widget
    }
)

# Display the heatmap title and interaction
display(HTML("<h2>Average Section Scores Over Time (Heatmap)</h2>"))
display(ui_heatmap, out_heatmap)

In [ ]:
# @title Data Quality Check

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets # Import ipywidgets
from ipywidgets import interact # Import interact

# Data Importation (re-using URL from previous cells for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Convert 'delivery_time' and 'date_submitted' to datetime, coercing errors to NaT
# Added yearfirst=True to aid in more robust date parsing, especially if mixed formats are present,
# as inferring was problematic (UserWarning: Could not infer format)
df['delivery_time'] = pd.to_datetime(df['delivery_time'], errors='coerce', yearfirst=True)
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce', yearfirst=True)
# Derive observation_date for filtering (only the date part)
df['observation_date'] = df['date_submitted'].dt.date

# Create dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

# Get unique observation dates, sort them, and add "All" option
valid_observation_dates = df['observation_date'].dropna().unique().tolist()
date_widget = widgets.Dropdown(
    options=["All"] + sorted(valid_observation_dates),
    value="All",
    description="Observation Date:"
)

observer_widget = widgets.Dropdown(
    options=["All"] + sorted(df['observer_name'].dropna().unique().tolist()),
    value="All",
    description="Observer:"
)

# Define the plotting function that accepts filter values
def plot_average_time_diff_filtered(selected_county, selected_observation_date, selected_observer):
    filtered_df = df.copy()

    # Apply county filter
    if selected_county != "All":
        filtered_df = filtered_df[filtered_df['county'] == selected_county]

    # Apply observation date filter
    if selected_observation_date != "All":
        # Ensure the comparison is between date objects
        filtered_df = filtered_df[filtered_df['observation_date'] == selected_observation_date]

    # Apply observer filter
    if selected_observer != "All":
        filtered_df = filtered_df[filtered_df['observer_name'] == selected_observer]

    # Drop rows where 'delivery_time' is NaT, as these cannot be used for time difference calculations
    # and also ensure 'observer_name' is present.
    df_cleaned = filtered_df.dropna(subset=['delivery_time', 'observer_name']).copy()

    # Check if any data remains after all filtering
    if df_cleaned.empty:
        print("No data available for the selected filters.")
        plt.figure(figsize=(12, 7))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Sort by observer_name and delivery_time to ensure correct consecutive difference calculation
    df_cleaned = df_cleaned.sort_values(by=['observer_name', 'delivery_time'])

    # Calculate the time difference between consecutive observations for each observer
    # The .diff() method calculates the difference between a value and the previous value in the series.
    # Grouping by 'observer_name' ensures that differences are only calculated within the same observer's records.
    df_cleaned['time_diff'] = df_cleaned.groupby('observer_name')['delivery_time'].diff()

    # Convert timedelta to total seconds and then to hours for numerical comparison and plotting
    df_cleaned['time_diff_hours'] = df_cleaned['time_diff'].dt.total_seconds() / 3600

    # Calculate the average time difference in hours for each observer
    # Drop NaN values that result from the first observation for each observer (no prior observation to diff with)
    average_time_diff_per_observer = df_cleaned.groupby('observer_name')['time_diff_hours'].mean().dropna()

    # Sort the results for better visualization
    average_time_diff_per_observer = average_time_diff_per_observer.sort_values(ascending=False)

    # Check if there is data to plot after time diff calculation
    if average_time_diff_per_observer.empty:
        print("No data available to plot average time differences per observer for the selected filters.")
        plt.figure(figsize=(12, 7))
        plt.text(0.5, 0.5, 'No data available to plot average time differences.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 7))
    ax = plt.gca() # Get the current axes
    bars = plt.bar(
        average_time_diff_per_observer.index,
        average_time_diff_per_observer.values,
        color='skyblue' # A pleasant color for the bars
    )

    plt.title('Average Time Difference Between Consecutive Observations per Observer', fontsize=16)
    plt.xlabel('Observer Name', fontsize=12)
    plt.ylabel('Average Time Difference (Hours)', fontsize=12)
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability

    # Remove grid lines
    plt.grid(False)

    # Remove top and right borders, make left and bottom borders light grey
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('lightgrey')
    ax.spines['bottom'].set_color('lightgrey')

    # Add data labels on top of each bar
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2,
                 yval + 0.05,
                 f'{yval:.2f} hrs',
                 ha='center',
                 va='bottom',
                 fontsize=9)

    plt.tight_layout() # Adjust layout to prevent labels from overlapping
    plt.show()

# Interactive dashboard setup
ui = widgets.HBox([county_widget, date_widget, observer_widget])

# Use interact to link widgets to the plotting function
interact(
    plot_average_time_diff_filtered,
    selected_county=county_widget,
    selected_observation_date=date_widget,
    selected_observer=observer_widget
);

In [ ]:
# @title Quality Care Score by Facility

import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.pyplot as plt
import pandas as pd

#Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Dropdowns for filters
month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

def plot_quality_by_facility(month, county, cadre):

    # Apply filters
    filtered_df = df.copy()

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if cadre != "All":
        filtered_df = filtered_df[filtered_df['cadre'] == cadre]

    # Average score by facility
    facility_scores = (
        filtered_df
        .groupby('facility', dropna=False)['quality_care_score']
        .mean()
        .multiply(100)  # Convert to %
        .sort_values(ascending=False)
    )

    # Colors based on 80% threshold
    colors = [
        'purple' if v >= 80 else '#F4B400'
        for v in facility_scores
    ]

    # Plot
    plt.figure(figsize=(12, 6))
    ax = facility_scores.plot(
        kind='bar',
        color=colors,
        edgecolor='none'
    )

    plt.title(
        "Average Quality Care Score by Facility",
        fontsize=12,
        style='italic'
    )

    plt.ylabel("Average Score (%)", fontsize=10)
    plt.xlabel("")
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # 80% threshold line
    plt.axhline(
        y=80,
        color='red',
        linestyle='--',
        linewidth=1.5
    )

    plt.text(
        len(facility_scores) - 0.5,
        81,
        "80% Threshold",
        color='red',
        fontsize=9,
        ha='right',
        va='bottom',
        style='italic'
    )

    # Clean borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('grey')
    ax.spines['left'].set_color('grey')
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)

    # Data labels
    for i, v in enumerate(facility_scores):
        if not pd.isna(v):
            plt.text(
                i,
                max(v - 3, 2),
                f"{v:.1f}%",
                ha='center',
                va='top',
                fontsize=9,
                weight='bold',
                color='white'
            )

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_quality_by_facility,
    month=month_widget,
    county=county_widget,
    cadre=cadre_widget
)

In [ ]:
# @title Quality of Care Score by Section

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# ----------------------------
# WIDGETS (FILTERS)
# ----------------------------
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

cadre_widget = widgets.Dropdown(
    options=["All"] + sorted(df['cadre'].dropna().unique().tolist()),
    value="All",
    description="Cadre:"
)

# ----------------------------
# GLOBAL INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# ----------------------------
# INDICATORS
# ----------------------------
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']

emotive_indicators = ['uterine_massage_initiated', 'timely_uterine_massage',
                      'oxytocin_first_line', 'timely_oxytocin_administration',
                      'txa_administered', 'timely_txa_administration',
                      'iv_access_fluids', 'timely_iv_establishment']

# Technical Quality After Birth (29 indicators)
# = former "Technical Quality Immediately Following Birth" (21)
#   + former "Maternal Vitals & Documentation" (8)
tqab_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                   'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                   'method_blood_loss', 'proper_vdrape_use', 'maternal_bleeding_assessed',
                   'administer_vitamin_k', 'timely_vitamink_administration',
                   'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation',
                   'take_mother_bp', 'timely_mother_bp',
                   'take_mother_temperature', 'timely_mother_temp',
                   'take_mother_pulse', 'timely_mother_pulse',
                   'apgar_min1', 'apgar_min5']

infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']

lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# ----------------------------
# MAIN PLOT FUNCTION
# ----------------------------
def plot_scores(county, facility, month, cadre):

    dff = df.copy()

    if county != "All":
        dff = dff[dff["county"] == county]
    if facility != "All":
        dff = dff[dff["facility"] == facility]
    if month != "All":
        dff = dff[dff["month"] == month]
    if cadre != "All":
        dff = dff[dff["cadre"] == cadre]

    section_mapping = {
        "Birth Preparedness": birth_preparedness,
        "Technical Quality After Birth": tqab_indicators,
        "Postbirth Infection Prevention": infection_indicators,
        "Respectful Maternity Care": rmc_indicators,
        "Avoidance Harmful Practices": harm_indicators,
        "Partograph Review": partograph_indicators,
        "Labor Care Guide Review": lcg_indicators,
        "EMOTIVE": emotive_indicators
    }

    calculated_mean_scores = {}

    for section_name, indicators in section_mapping.items():
        if isinstance(indicators, str):
            calculated_mean_scores[section_name] = dff[indicators].mean()
        else:
            scores = []
            for col in indicators:
                if col in dff.columns:
                    scores.append(compute_score_unified(dff[col], col))
            calculated_mean_scores[section_name] = pd.Series(scores).mean()

    mean_scores = pd.Series(calculated_mean_scores).sort_values(ascending=False) * 100
    benchmark = 85  # Fixed benchmark at 85%

    colors = ['purple' if s >= benchmark else 'goldenrod' for s in mean_scores]

    plt.figure(figsize=(14, 7))
    ax = mean_scores.plot(kind='bar', color=colors)

    plt.title('Average Score by Section')
    plt.ylabel('Average Score (%)')
    plt.xlabel('Section')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Benchmark line
    plt.axhline(y=benchmark, color='red', linestyle='--', linewidth=1.5)

    # Benchmark label (fixed)
    ax.text(
        len(mean_scores) - 0.5 + 0.1,  # X-coordinate: slightly to the right of the last bar's edge
        benchmark,
        f"Benchmark: {benchmark:.0f}%",
        color='red',
        fontsize=10,
        ha='left',
        va='center'
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    for i, v in enumerate(mean_scores):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# ----------------------------
# INTERACTIVE WIDGET DISPLAY
# ----------------------------
ui = widgets.HBox([county_widget, facility_widget, month_widget, cadre_widget])

out = widgets.interactive_output(
    plot_scores,
    {
        "county": county_widget,
        "facility": facility_widget,
        "month": month_widget,
        "cadre": cadre_widget
    }
)

display(ui, out)

In [ ]:
# @title Quality of Care Score Card by Section Over Time

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Convert date + derive month period
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# WIDGETS (FILTERS)
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')
cadre_slicer = Slicer(df, 'cadre', 'Cadre')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# GLOBAL INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# ----------------------------
# INDICATORS
# ----------------------------
birth_preparedness = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']

emotive_indicators = ['uterine_massage_initiated', 'timely_uterine_massage',
                      'oxytocin_first_line', 'timely_oxytocin_administration',
                      'txa_administered', 'timely_txa_administration',
                      'iv_access_fluids', 'timely_iv_establishment']

# Technical Quality After Birth (29 indicators)
# = former "Technical Quality Immediately Following Birth" (21)
#   + former "Maternal Vitals & Documentation" (8)
tqab_indicators = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin', 'adequate_skin_to_skin_time',
                   'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination', 'uterine_tone_assessed', 'laceration_check',
                   'method_blood_loss', 'proper_vdrape_use', 'maternal_bleeding_assessed',
                   'administer_vitamin_k', 'timely_vitamink_administration',
                   'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation',
                   'take_mother_bp', 'timely_mother_bp',
                   'take_mother_temperature', 'timely_mother_temp',
                   'take_mother_pulse', 'timely_mother_pulse',
                   'apgar_min1', 'apgar_min5']

infection_indicators = ['sharps_disposal', 'waste_disposal']
rmc_indicators = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']
harm_indicators = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

partograph_indicators = ['partogragh_initiation', 'partograph_half_hr', 'fetal_heart_correct', 'maternal_pulse_correct', 'bp_4hrs', 'partograph_summary_filled', 'delivery_method_filled',
                         'blood_loss_estimation', 'partograph_accuracy_concern']

lcg_indicators = ['lcg_initiated_correct', 'contractions_monitored_1st', 'fhr_monitored_1st', 'bp_monitored_recorded', 'contractions_monitored_2nd', 'fhr_monitored_2nd', 'delivery_time_documented',
                  'delivery_method_documented', 'ebl_documented_correctly', 'lcg_accuracy_concerns']

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()

    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count

    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()

    elif col_name in harm_indicators + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()

    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return yes_count / denom

# ----------------------------
# STATUS + TREND FUNCTIONS
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, in percentage points (pp)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

def section_score_for_period(data, indicators, period):
    """Mean indicator score (0-100) for one section in one month."""
    sub = data[data['month_period'] == period]
    scores = [compute_score_unified(sub[c], c) for c in indicators]
    scores = [s for s in scores if not pd.isna(s)]
    return np.mean(scores) * 100 if scores else np.nan

# ----------------------------
# MAIN SCORECARD FUNCTION
# ----------------------------
def build_section_scorecard(counties, facilities, cadres, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, "county", counties)
    base_df = apply_slicer(base_df, "facility", facilities)
    base_df = apply_slicer(base_df, "cadre", cadres)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    range_df = base_df.copy()
    if start_month is not None:
        range_df = range_df[range_df['month_period'] >= start_month]
    if end_month is not None:
        range_df = range_df[range_df['month_period'] <= end_month]

    if range_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    section_mapping = {
        "Birth Preparedness": birth_preparedness,
        "Technical Quality After Birth": tqab_indicators,
        "Postbirth Infection Prevention": infection_indicators,
        "Respectful Maternity Care": rmc_indicators,
        "Avoidance Harmful Practices": harm_indicators,
        "Partograph Review": partograph_indicators,
        "Labor Care Guide Review": lcg_indicators,
        "EMOTIVE": emotive_indicators
    }

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(range_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')   # current month
    prev_month = trend_month - 1

    trend_col = "Trend"

    rows = []

    for section_name, indicators in section_mapping.items():
        if not all(col in range_df.columns for col in indicators):
            continue

        indicator_row = {"Section": section_name}

        # Overall score for the selected filters + month range
        overall_scores_list = []
        for col in indicators:
            score = compute_score_unified(range_df[col], col)
            if not pd.isna(score):
                overall_scores_list.append(score)
        overall_mean_score = np.mean(overall_scores_list) * 100 if overall_scores_list else np.nan
        indicator_row["Score"] = f"{overall_mean_score:.1f}%" if not pd.isna(overall_mean_score) else "N/A"

        # Monthly columns (within the range)
        for m in months_to_display:
            s = section_score_for_period(range_df, indicators, m)
            indicator_row[m.strftime('%b %Y')] = f"{s:.1f}%" if not pd.isna(s) else "N/A"

        # Trend uses base_df (not month-limited) so the previous month is always available
        current_score = section_score_for_period(base_df, indicators, trend_month)
        previous_score = section_score_for_period(base_df, indicators, prev_month)

        indicator_row["Status"] = status_icon(overall_mean_score)
        indicator_row[trend_col] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    scorecard_df = pd.DataFrame(rows)

    # Convert 'Score' column to numeric for sorting, handling 'N/A'
    scorecard_df['Sort_Score'] = scorecard_df['Score'].str.replace('%', '').replace('N/A', np.nan).astype(float)
    scorecard_df = scorecard_df.sort_values(by='Sort_Score', ascending=True).drop(columns=['Sort_Score'])

    # Reorder columns: Section, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Section", "Score"] + [m.strftime('%b %Y') for m in months_to_display] + ["Status", trend_col]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# INTERACTIVE DISPLAY
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_section_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            cadre_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
cadre_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, cadre_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Key QuIPS Proposed Indicators
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Preprocess date to enable month filtering and trend calculation
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month_period'] = df['date_submitted'].dt.to_period('M')  # For monthly calculations

# Ensure '_uuid' is available for unique delivery counting, if not, use index for count
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# GLOBAL INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A", "Resuscitation required", "Not taken"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# ----------------------------
# INDICATOR DEFINITIONS (from other scorecards)
# ----------------------------
birth_preparedness_cols = ['hand_hygiene', 'ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap', 'nnr_equipment_preparation', 'uterotonic_preparation']

# Technical Quality Immediately Following Birth (21 indicators)
tqifb_cols = ['ruleout_twin', 'uterotonic_administered', 'timely_uterotonic_administration', 'drying_stimulation', 'discard_wet_towel', 'skin_to_skin',
              'adequate_skin_to_skin_time', 'timely_cord_clamp', 'sterilized_blade_clamp', 'newborn_kept_warm', 'controlled_cord_traction', 'placenta_examination',
              'uterine_tone_assessed', 'laceration_check', 'method_blood_loss', 'proper_vdrape_use', 'maternal_bleeding_assessed',
              'administer_vitamin_k', 'timely_vitamink_administration',
              'provider_initiate_breastfeeding', 'timely_breastfeeding_initiation']

infection_cols = ['sharps_disposal', 'waste_disposal']

rmc_cols = ['birth_companion', 'provider_explanations', 'spoken_to_directly', 'spoken_to_kindly', 'rmc_no_disrespect_to_mother']

harm_avoidance_cols = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Specific columns for derived indicators
newborn_essential_care_cols = ['drying_stimulation', 'skin_to_skin', 'timely_cord_clamp', 'timely_breastfeeding_initiation']
timely_vitals_cols = ['timely_mother_bp', 'timely_mother_temp', 'timely_mother_pulse']

# ----------------------------
# SCORE COMPUTATION FUNCTIONS
# ----------------------------
def compute_indicator_score(series, col_name, invalid_vals=invalid_values):
    clean = series[~series.isin(invalid_vals)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_vals)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in harm_avoidance_cols + ['partograph_accuracy_concern', 'lcg_accuracy_concerns']:
        # For harmful practices or concerns, 'No' is the desirable outcome
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return (yes_count / denom) * 100

def calculate_section_score(df_filtered, indicator_cols, invalid_vals=invalid_values):
    if df_filtered.empty or not indicator_cols:
        return np.nan

    scores = []
    for col in indicator_cols:
        if col in df_filtered.columns:
            # Need to explicitly pass col to compute_indicator_score for special logic
            score = compute_indicator_score(df_filtered[col], col, invalid_vals)
            if not pd.isna(score):
                scores.append(score)

    if not scores:
        return np.nan
    return np.mean(scores)

# ----------------------------
# STATUS + TREND FUNCTIONS
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# WIDGETS (FILTERS)
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')
cadre_slicer = Slicer(df, 'cadre', 'Cadre')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_quips_scorecard(counties, facilities, cadres, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)
    base_df = apply_slicer(base_df, 'cadre', cadres)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Helper to calculate scores for a given period DataFrame
    def get_score_for_df_period(df_sub_period, indicator_name_key, indicator_cols_list=None, threshold=None, is_100_percent=False):
        if df_sub_period.empty:
            return np.nan

        period_total_deliveries = df_sub_period['_uuid'].nunique()
        if period_total_deliveries == 0:
            return np.nan

        if indicator_name_key == "mentees_delivery":
            mentee_deliveries = df_sub_period[df_sub_period['cadre'] == 'Enrolled jh mentee']['_uuid'].nunique() if 'cadre' in df_sub_period.columns else 0
            return (mentee_deliveries / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "section_score":
            return calculate_section_score(df_sub_period, indicator_cols_list, invalid_values)
        elif indicator_name_key == "overall_qoc":
            return df_sub_period['quality_care_score'].mean() * 100 if 'quality_care_score' in df_sub_period.columns else np.nan
        elif indicator_name_key == "lcg_usage":
            lcg_usage_count = df_sub_period[df_sub_period['lcg_initiated_correct'] == 'Yes']['_uuid'].nunique() if 'lcg_initiated_correct' in df_sub_period.columns else 0
            return (lcg_usage_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "newborn_essential_care":
            newborn_care_met_count = 0
            if all(col in df_sub_period.columns for col in newborn_essential_care_cols):
                temp_df = df_sub_period[newborn_essential_care_cols].copy()
                for col in newborn_essential_care_cols:
                    temp_df[col] = temp_df[col].apply(lambda x: x if x not in invalid_values else np.nan)
                newborn_care_met_uuid = df_sub_period.loc[
                    (temp_df['drying_stimulation'] == 'Yes') &
                    (temp_df['skin_to_skin'] == 'Yes') &
                    (temp_df['timely_cord_clamp'] == 'Yes') &
                    (temp_df['timely_breastfeeding_initiation'] == 'Yes')
                ]['_uuid'].nunique()
                newborn_care_met_count = newborn_care_met_uuid
            return (newborn_care_met_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "vitk_admin":
            vitk_count = df_sub_period[df_sub_period['timely_vitamink_administration'] == 'Yes']['_uuid'].nunique() if 'timely_vitamink_administration' in df_sub_period.columns else 0
            return (vitk_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "timely_uterotonic_administration":
            timely_uterotonic_count = df_sub_period[df_sub_period['timely_uterotonic_administration'] == 'Yes']['_uuid'].nunique() if 'timely_uterotonic_administration' in df_sub_period.columns else 0
            return (timely_uterotonic_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "delayed_cord_clamp":
            delayed_cord_clamp_count = df_sub_period[df_sub_period['timely_cord_clamp'] == 'Yes']['_uuid'].nunique() if 'timely_cord_clamp' in df_sub_period.columns else 0
            return (delayed_cord_clamp_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "timely_vitals":
            timely_vitals_count = 0
            if all(col in df_sub_period.columns for col in timely_vitals_cols):
                temp_df = df_sub_period[timely_vitals_cols].copy()
                for col in timely_vitals_cols:
                    temp_df[col] = temp_df[col].apply(lambda x: x if x not in invalid_values else np.nan)
                timely_vitals_met_uuid = df_sub_period.loc[
                    (temp_df['timely_mother_bp'] == 'Yes') &
                    (temp_df['timely_mother_temp'] == 'Yes') &
                    (temp_df['timely_mother_pulse'] == 'Yes')
                ]['_uuid'].nunique()
                timely_vitals_count = timely_vitals_met_uuid
            return (timely_vitals_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "timely_breastfeeding":
            timely_breastfeeding_count = df_sub_period[df_sub_period['timely_breastfeeding_initiation'] == 'Yes']['_uuid'].nunique() if 'timely_breastfeeding_initiation' in df_sub_period.columns else 0
            return (timely_breastfeeding_count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan
        elif indicator_name_key == "section_threshold_count":
            # Helper to count deliveries meeting a section score threshold
            def count_deliveries_by_section_score_threshold(df_sub, indicator_cols_list, threshold, is_100_percent_inner=False):
                deliveries_met_threshold = 0
                if df_sub.empty or not indicator_cols_list:
                    return 0  # Return 0 for count, not nan for score

                grouped_deliveries = df_sub.groupby('_uuid')

                for uuid, delivery_df in grouped_deliveries:
                    delivery_scores = []
                    for col in indicator_cols_list:
                        if col in delivery_df.columns:
                            score = compute_indicator_score(delivery_df[col], col, invalid_values)
                            if not pd.isna(score):
                                delivery_scores.append(score)

                    if delivery_scores:
                        avg_delivery_score = np.mean(delivery_scores)
                        if is_100_percent_inner:
                            # All *observed and valid* indicators for this delivery
                            # in this section scored 100%.
                            if avg_delivery_score == threshold:
                                deliveries_met_threshold += 1
                        else:
                            if avg_delivery_score >= threshold:
                                deliveries_met_threshold += 1
                return deliveries_met_threshold

            count = count_deliveries_by_section_score_threshold(df_sub_period, indicator_cols_list, threshold, is_100_percent)
            return (count / period_total_deliveries) * 100 if period_total_deliveries > 0 else np.nan

        return np.nan

    all_indicators_spec = [
        {"key": "mentees_delivery", "label": "% of deliveries by JH enrolled mentees"},
        {"key": "section_threshold_count", "label": "% of deliveries with Birth Preparedness Score of 100%", "cols": birth_preparedness_cols, "threshold": 100, "is_100_percent": True},
        {"key": "section_threshold_count", "label": "% of deliveries with Respectful Maternity Care (RMC) Score of 100%", "cols": rmc_cols, "threshold": 100, "is_100_percent": True},
        {"key": "section_threshold_count", "label": "% of deliveries with Technical Quality Immediately Following Birth Score \u2265 90%", "cols": tqifb_cols, "threshold": 90, "is_100_percent": False},
        {"key": "section_threshold_count", "label": "% of deliveries with Post-Birth Infection Prevention Score of 100%", "cols": infection_cols, "threshold": 100, "is_100_percent": True},
        {"key": "section_threshold_count", "label": "% of deliveries with Avoidance of Harmful Practices Score of 100%", "cols": harm_avoidance_cols, "threshold": 100, "is_100_percent": True},
        {"key": "lcg_usage", "label": "% of deliveries using Labor Care Guide"},
        {"key": "overall_qoc", "label": "Overall Intrapartum Quality of Care Score"},
        {"key": "newborn_essential_care", "label": "% of live deliveries with all essential newborn care components"},
        {"key": "vitk_admin", "label": "% of live deliveries with timely administration of Vitamin K"},
        {"key": "timely_uterotonic_administration", "label": "% of live deliveries with timely administration of uterotonics"},
        {"key": "delayed_cord_clamp", "label": "% of live deliveries with delayed cord clamping"},
        {"key": "timely_vitals", "label": "% of deliveries with timely recording of vitals (BP, Temp, Pulse)"},
        {"key": "timely_breastfeeding", "label": "% of live deliveries with timely initiation of breastfeeding"}
    ]

    # Months used for the Trend column. Taken from base_df (not month-limited)
    # so the previous month is available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    results = []

    for indicator_spec in all_indicators_spec:
        indicator_row = {"Indicator": indicator_spec["label"]}

        spec_kwargs = dict(
            indicator_cols_list=indicator_spec.get("cols"),
            threshold=indicator_spec.get("threshold"),
            is_100_percent=indicator_spec.get("is_100_percent", False)
        )

        # Overall score for the selected filters + month range
        overall_score = get_score_for_df_period(full_filtered_df, indicator_spec["key"], **spec_kwargs)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        for current_month_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_month_period]
            score = get_score_for_df_period(df_for_month, indicator_spec["key"], **spec_kwargs)
            indicator_row[f"{current_month_period.strftime('%b %Y')}"] = f"{score:.1f}%" if not pd.isna(score) else "N/A"

        # Status based on the overall score
        indicator_row["Status"] = status_icon(overall_score)

        # Trend: 'To' month vs the month before it
        current_score = get_score_for_df_period(df_trend_month, indicator_spec["key"], **spec_kwargs)
        previous_score = get_score_for_df_period(df_prev_month, indicator_spec["key"], **spec_kwargs)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)

        results.append(indicator_row)

    scorecard_df = pd.DataFrame(results)

    # Convert 'Score' column to numeric for sorting, handling 'N/A'
    scorecard_df['Sort_Score'] = scorecard_df['Score'].str.replace('%', '').replace('N/A', np.nan).astype(float)
    scorecard_df = scorecard_df.sort_values(by='Sort_Score', ascending=True)
    scorecard_df = scorecard_df.drop(columns=['Sort_Score'])

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# INTERACTIVE DISPLAY
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_quips_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            cadre_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
cadre_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, cadre_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Birth Preparedness Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Convert date + derive month period
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
indicator_labels = {
    'hand_hygiene': 'Hand Hygiene',
    'ppe_gloves': 'PPE Gloves',
    'ppe_mask': 'PPE Mask',
    'ppe_gown': 'PPE Gown',
    'ppe_boots': 'PPE Boots',
    'ppe_goggles': 'PPE Goggles',
    'ppe_cap': 'PPE Cap',
    'nnr_equipment_preparation': 'Preparation of NNR Equipment',
    'uterotonic_preparation': 'Uterotonic Preparation'
}

birth_preparedness = list(indicator_labels.keys())

# ----------------------------
# WIDGETS (FILTERS)
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# VALID RESPONSE FILTER
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A", "Resuscitation required", "Not taken"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# ----------------------------
# SCORE COMPUTATION FUNCTION
# ----------------------------
def compute_indicator_score(series, col_name, invalid_vals=invalid_values):
    clean = series[~series.isin(invalid_vals)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_vals)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down'] + ['partograph_accuracy_concern', 'lcg_accuracy_concerns']:
        # For harmful practices or concerns, 'No' is the desirable outcome
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return (yes_count / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col in birth_preparedness:
        if col not in df.columns:
            continue

        indicator_row = {"Indicator": indicator_labels.get(col, col)}

        # Score for the selected filters + month range
        overall_score = compute_indicator_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_indicator_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_indicator_score(df_trend_month[col], col)
        previous_score = compute_indicator_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    scorecard_df = pd.DataFrame(rows)

    # Convert 'Score' column to numeric for sorting, handling 'N/A'
    scorecard_df['Sort_Score'] = scorecard_df['Score'].str.replace('%', '').replace('N/A', np.nan).astype(float)
    scorecard_df = scorecard_df.sort_values(by='Sort_Score', ascending=True)
    scorecard_df = scorecard_df.drop(columns=['Sort_Score'])  # Drop the temporary sorting column

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# INTERACTIVE DISPLAY
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Technical Quality After Birth Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Robust datetime parsing
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
# Technical Quality After Birth (29 indicators)
# = former "Technical Quality Immediately Following Birth" (21)
#   + former "Maternal Vitals & Documentation" (8)
tqab_indicators = {
    'ruleout_twin': 'Ruling out second twin',
    'uterotonic_administered': 'Uterotonic administration',
    'timely_uterotonic_administration': 'Timely administration of uterotonics',
    'drying_stimulation': 'Baby dried and stimulated',
    'discard_wet_towel': 'Wet towel discarded',
    'skin_to_skin': 'Skin-to-skin initiated',
    'adequate_skin_to_skin_time': 'Adequate skin-to-skin duration',
    'timely_cord_clamp': 'Timely cord clamping',
    'sterilized_blade_clamp': 'Use of sterilized blade for cord cut',
    'newborn_kept_warm': 'Newborn kept warm for resuscitation or stimulation',
    'controlled_cord_traction': 'Use of controlled cord traction placenta delivery',
    'placenta_examination': 'Placenta examination postbirth',
    'uterine_tone_assessed': 'Uterine tone assessed',
    'laceration_check': 'Birth canal checked for lacerations',
    'method_blood_loss': 'Objective method of estimating blood loss used',
    'proper_vdrape_use': 'V drape used appropriately',
    'maternal_bleeding_assessed': 'Maternal bleeding assessed',
    'administer_vitamin_k': 'Vitamin K administered',
    'timely_vitamink_administration': 'Timely administration of Vitamin K',
    'provider_initiate_breastfeeding': 'Breastfeeding initiated',
    'timely_breastfeeding_initiation': 'Timely initiation of breastfeeding',
    # --- Moved in from Maternal Vitals & Documentation ---
    'take_mother_bp': 'Maternal blood pressure taken',
    'timely_mother_bp': 'Timely maternal blood pressure measurement',
    'take_mother_temperature': 'Maternal temperature taken',
    'timely_mother_temp': 'Timely maternal temperature measurement',
    'take_mother_pulse': 'Maternal pulse taken',
    'timely_mother_pulse': 'Timely maternal pulse measurement',
    'apgar_min1': 'Apgar score documented at 1 minute',
    'apgar_min5': 'Apgar score documented at 5 minutes'
}

# ----------------------------
# RULES
# ----------------------------
blood_loss_valid = [
    "Measure with a calibrated jar",
    "V drape",
    "Weighing of bloodsoaked materials"
]

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A", "Resuscitation required", "Not taken"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()


harm_avoidance_cols = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_indicator_score(series, col_name, invalid_vals=invalid_values):
    clean = series[~series.isin(invalid_vals)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_vals)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in harm_avoidance_cols + ['partograph_accuracy_concern', 'lcg_accuracy_concerns']:
        # For harmful practices or concerns, 'No' is the desirable outcome
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return (yes_count / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_tqab_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in tqab_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_indicator_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_indicator_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_indicator_score(df_trend_month[col], col)
        previous_score = compute_indicator_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Convert 'Score' column to numeric for sorting, handling 'N/A'
    scorecard_df['Sort_Score'] = scorecard_df['Score'].str.replace('%', '').replace('N/A', np.nan).astype(float)
    scorecard_df = scorecard_df.sort_values(by='Sort_Score', ascending=True)
    scorecard_df = scorecard_df.drop(columns=['Sort_Score'])  # Drop the temporary sorting column

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_tqab_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Uterotonic Administered Scorecard (% use by category)

import pandas as pd
import numpy as np
import ipywidgets as widgets
from collections import Counter
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# SETTINGS
# ----------------------------
FIELD = 'uterotonic_administered'
MULTI_SELECT = False   # set True if more than one uterotonic can be selected per delivery
                       # (Kobo exports multiple selections as space-separated values)

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

if '_uuid' not in df.columns:
    df['_uuid'] = df.index

df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INVALID VALUES (excluded from the denominator)
# ----------------------------
invalid_values = {"", "unable to observe", "not applicable", "na", "n/a", "nan"}

# ----------------------------
# CATEGORY COUNTS
# ----------------------------
def category_counts(series):
    """Returns (Counter of category -> deliveries, number of deliveries with a valid answer)."""
    s = series.dropna().astype(str).str.strip()
    s = s[~s.str.lower().isin(invalid_values)]

    if MULTI_SELECT:
        rows = [set(x.split()) for x in s]
    else:
        rows = [{x} for x in s]

    counts = Counter()
    for r in rows:
        counts.update(r)
    return counts, len(rows)

def pct_of(counts, n, category):
    return np.nan if n == 0 else counts.get(category, 0) / n * 100

def fmt_pct(p):
    return "N/A" if pd.isna(p) else f"{p:.1f}%"

# ----------------------------
# TREND (neutral colours: whether up is good depends on the category)
# ----------------------------
def trend_icon(current, previous):
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:#1f77b4; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:#ff7f0e; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_uterotonic_scorecard(counties, facilities, start_month, end_month):

    if FIELD not in df.columns:
        display(HTML(f"<h3>Column '{FIELD}' not found in the data.</h3>"))
        return

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())
    if not months_to_display:
        display(HTML("<h3>No monthly data available.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')   # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    cur_counts, cur_n = category_counts(
        base_df[base_df['month_period'] == trend_month][FIELD])
    prev_counts, prev_n = category_counts(
        base_df[base_df['month_period'] == prev_month][FIELD])

    # ---- Overall + per-month counts ----
    overall_counts, overall_n = category_counts(full_filtered_df[FIELD])

    if overall_n == 0:
        display(HTML("<h3>No valid responses for the selected filters.</h3>"))
        return

    month_data = {}
    for m in months_to_display:
        month_data[m] = category_counts(full_filtered_df[full_filtered_df['month_period'] == m][FIELD])

    # Categories sorted by overall use (highest first)
    categories = [c for c, _ in overall_counts.most_common()]

    rows = []
    for cat in categories:
        row = {"Uterotonic": cat}
        row["Overall"] = fmt_pct(pct_of(overall_counts, overall_n, cat))

        for m in months_to_display:
            counts, n = month_data[m]
            row[m.strftime('%b %Y')] = fmt_pct(pct_of(counts, n, cat))

        row["Trend"] = trend_icon(
            pct_of(cur_counts, cur_n, cat),
            pct_of(prev_counts, prev_n, cat)
        )
        rows.append(row)

    scorecard_df = pd.DataFrame(rows)

    ordered_columns = ["Uterotonic", "Overall"] + [m.strftime('%b %Y') for m in months_to_display] + ["Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_uterotonic_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Method of Blood Loss Measurement Scorecard (% use by category)

import pandas as pd
import numpy as np
import ipywidgets as widgets
from collections import Counter
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# SETTINGS
# ----------------------------
FIELD = 'method_blood_loss'
ROW_LABEL = 'Method of blood loss measurement'
MULTI_SELECT = False   # set True if more than one method can be selected per delivery
                       # (Kobo exports multiple selections as space-separated values)

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

if '_uuid' not in df.columns:
    df['_uuid'] = df.index

df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INVALID VALUES (excluded from the denominator)
# ----------------------------
invalid_values = {"", "unable to observe", "not applicable", "na", "n/a", "nan"}

# ----------------------------
# CATEGORY COUNTS
# ----------------------------
def category_counts(series):
    """Returns (Counter of category -> deliveries, number of deliveries with a valid answer)."""
    s = series.dropna().astype(str).str.strip()
    s = s[~s.str.lower().isin(invalid_values)]

    if MULTI_SELECT:
        rows = [set(x.split()) for x in s]
    else:
        rows = [{x} for x in s]

    counts = Counter()
    for r in rows:
        counts.update(r)
    return counts, len(rows)

def pct_of(counts, n, category):
    return np.nan if n == 0 else counts.get(category, 0) / n * 100

def fmt_pct(p):
    return "N/A" if pd.isna(p) else f"{p:.1f}%"

# ----------------------------
# TREND (neutral colours: whether up is good depends on the category)
# ----------------------------
def trend_icon(current, previous):
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:#1f77b4; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:#ff7f0e; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_blood_loss_scorecard(counties, facilities, start_month, end_month):

    if FIELD not in df.columns:
        display(HTML(f"<h3>Column '{FIELD}' not found in the data.</h3>"))
        return

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())
    if not months_to_display:
        display(HTML("<h3>No monthly data available.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')   # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    cur_counts, cur_n = category_counts(
        base_df[base_df['month_period'] == trend_month][FIELD])
    prev_counts, prev_n = category_counts(
        base_df[base_df['month_period'] == prev_month][FIELD])

    # ---- Overall + per-month counts ----
    overall_counts, overall_n = category_counts(full_filtered_df[FIELD])

    if overall_n == 0:
        display(HTML("<h3>No valid responses for the selected filters.</h3>"))
        return

    month_data = {}
    for m in months_to_display:
        month_data[m] = category_counts(full_filtered_df[full_filtered_df['month_period'] == m][FIELD])

    # Categories sorted by overall use (highest first)
    categories = [c for c, _ in overall_counts.most_common()]

    rows = []
    for cat in categories:
        row = {ROW_LABEL: cat}
        row["Overall"] = fmt_pct(pct_of(overall_counts, overall_n, cat))

        for m in months_to_display:
            counts, n = month_data[m]
            row[m.strftime('%b %Y')] = fmt_pct(pct_of(counts, n, cat))

        row["Trend"] = trend_icon(
            pct_of(cur_counts, cur_n, cat),
            pct_of(prev_counts, prev_n, cat)
        )
        rows.append(row)

    scorecard_df = pd.DataFrame(rows)

    ordered_columns = [ROW_LABEL, "Overall"] + [m.strftime('%b %Y') for m in months_to_display] + ["Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_blood_loss_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Maternal Vitals & Documentation Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Robust datetime parsing
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
# Maternal Vitals & Documentation (8 indicators)
vitals_indicators = {
    'take_mother_bp': 'Mother BP taken',
    'timely_mother_bp': 'Timely taking of mother BP',
    'take_mother_temperature': 'Mother temperature taken',
    'timely_mother_temp': 'Timely taking of mother temperature',
    'take_mother_pulse': 'Mother pulse taken',
    'timely_mother_pulse': 'Timely taking of mother pulse',
    'apgar_min1': 'APGAR score at 1 minute',
    'apgar_min5': 'APGAR score at 5 minutes'
}

# ----------------------------
# RULES
# ----------------------------
blood_loss_valid = [
    "Measure with a calibrated jar",
    "V drape",
    "Weighing of bloodsoaked materials"
]

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A", "Resuscitation required", "Not taken"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()


harm_avoidance_cols = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_indicator_score(series, col_name, invalid_vals=invalid_values):
    clean = series[~series.isin(invalid_vals)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_vals)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in harm_avoidance_cols + ['partograph_accuracy_concern', 'lcg_accuracy_concerns']:
        # For harmful practices or concerns, 'No' is the desirable outcome
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return (yes_count / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_vitals_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in vitals_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_indicator_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_indicator_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_indicator_score(df_trend_month[col], col)
        previous_score = compute_indicator_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Convert 'Score' column to numeric for sorting, handling 'N/A'
    scorecard_df['Sort_Score'] = scorecard_df['Score'].str.replace('%', '').replace('N/A', np.nan).astype(float)
    scorecard_df = scorecard_df.sort_values(by='Sort_Score', ascending=True)
    scorecard_df = scorecard_df.drop(columns=['Sort_Score'])  # Drop the temporary sorting column

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_vitals_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title EMOTIVE Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Robust datetime parsing
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
emotive_indicators = {
    'uterine_massage_initiated': 'Uterine massage initiated',
    'timely_uterine_massage': 'Timely uterine massage',
    'oxytocin_first_line': 'Oxytocin first-line administered',
    'timely_oxytocin_administration': 'Timely administration of oxytocin',
    'txa_administered': 'Tranexamic Acid (TXA) administered',
    'timely_txa_administration': 'Timely administration of TXA',
    'iv_access_fluids': 'IV access established',
    'timely_iv_establishment': 'Timely establishment of IV access'
}

# ----------------------------
# RULES
# ----------------------------
blood_loss_valid = [
    "Measure with a calibrated jar",
    "V drape",
    "Weighing of bloodsoaked materials"
]

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A", "Resuscitation required", "Not taken"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()


harm_avoidance_cols = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_indicator_score(series, col_name, invalid_vals=invalid_values):
    clean = series[~series.isin(invalid_vals)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_vals)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in harm_avoidance_cols + ['partograph_accuracy_concern', 'lcg_accuracy_concerns']:
        # For harmful practices or concerns, 'No' is the desirable outcome
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count
    if denom == 0:
        return np.nan

    return (yes_count / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_emotive_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in emotive_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_indicator_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_indicator_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_indicator_score(df_trend_month[col], col)
        previous_score = compute_indicator_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No EMOTIVE indicator columns found in the data.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Convert 'Score' column to numeric for sorting, handling 'N/A'
    scorecard_df['Sort_Score'] = scorecard_df['Score'].str.replace('%', '').replace('N/A', np.nan).astype(float)
    scorecard_df = scorecard_df.sort_values(by='Sort_Score', ascending=True)
    scorecard_df = scorecard_df.drop(columns=['Sort_Score'])  # Drop the temporary sorting column

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_emotive_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Infection Prevention Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Robust datetime handling
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
infection_indicators = [
    'sharps_disposal',
    'waste_disposal'
]

# ----------------------------
# INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()


# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_score(series):

    clean = series[~series.isin(invalid_values)].dropna()
    clean = normalize(clean)
    clean_lower = clean.str.lower()

    yes = (clean_lower == "yes").sum()
    no = (clean_lower == "no").sum()

    denom = yes + no

    if denom == 0:
        return np.nan

    return (yes / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"


def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_infection_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col in infection_indicators:

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": col.replace("_", " ").title()}

        # Score for the selected filters + month range
        overall_score = compute_score(full_filtered_df[col])
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_score(df_for_month[col])
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_score(df_trend_month[col])
        previous_score = compute_score(df_prev_month[col])

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_infection_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Respectful Maternity Care Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Robust datetime parsing
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
rmc_indicators = {
    'birth_companion': 'Birth companion present',
    'provider_explanations': 'Provider explained process to mother clearly throughout',
    'spoken_to_directly': 'Provider spoke to the mother directly throughout',
    'spoken_to_kindly': 'Provider spoke to the mother kindly throughout',
    'rmc_no_disrespect_to_mother': 'Mother was treated with respect throughout'
}

# ----------------------------
# INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_score(series, col):

    clean = series[~series.isin(invalid_values)].dropna()
    clean = normalize(clean)

    if col in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:

        clean_lower = clean.str.lower()

        yes = clean_lower.isin(["yes", "always"]).sum()
        no = clean_lower.isin(["no", "sometimes", "rarely", "never"]).sum()

    else:
        clean_lower = clean.str.lower()
        yes = (clean_lower == "yes").sum()
        no = (clean_lower == "no").sum()

    denom = yes + no

    if denom == 0:
        return np.nan

    return (yes / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_rmc_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in rmc_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_score(df_trend_month[col], col)
        previous_score = compute_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_rmc_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Avoidance of Harmful Practices Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Robust datetime handling
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
harm_indicators = {
    'fundal_pressure': 'No instance of fundal pressure',
    'perineum_stretching': 'No instance of perineum stretching',
    'newborn_slapped': 'No instance of newborn slapping',
    'newborn_upside_down': 'No instance of newborn being held upside down'
}

# ----------------------------
# INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

# ----------------------------
# SCORE FUNCTION (NO = SAFE)
# ----------------------------
def compute_score(series):

    clean = series[~series.isin(invalid_values)].dropna()
    clean = normalize(clean)

    clean_lower = clean.str.lower()

    no_safe = (clean_lower == "no").sum()
    yes_harm = (clean_lower == "yes").sum()

    denom = no_safe + yes_harm

    if denom == 0:
        return np.nan

    return (no_safe / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_harm_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in harm_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_score(full_filtered_df[col])
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_score(df_for_month[col])
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_score(df_trend_month[col])
        previous_score = compute_score(df_prev_month[col])

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_harm_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Harmful Practices Scorecard (% of deliveries with instance)

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

if '_uuid' not in df.columns:
    df['_uuid'] = df.index

df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
harm_indicators = {
    'fundal_pressure': '% deliveries with fundal pressure',
    'perineum_stretching': '% deliveries with perineum stretching',
    'newborn_slapped': '% deliveries with newborn slapping',
    'newborn_upside_down': '% deliveries with newborn held upside down'
}

# ----------------------------
# YES / OBSERVED COUNTS
# ----------------------------
def yes_and_total(series):
    """Returns (yes_count, observed_count). Observed = Yes + No only;
    'Unable to observe', 'Not applicable', blanks etc. are excluded."""
    clean = series.dropna().astype(str).str.strip().str.lower()
    yes = int((clean == "yes").sum())
    no = int((clean == "no").sum())
    return yes, yes + no

def pct(series):
    yes, total = yes_and_total(series)
    return np.nan if total == 0 else yes / total * 100

def fmt_cell(series):
    yes, total = yes_and_total(series)
    if total == 0:
        return "N/A"
    return f"{yes / total * 100:.1f}%"

# ----------------------------
# STATUS + TREND (lower % = better)
# ----------------------------
def status_icon(p):
    if pd.isna(p):
        return "⚪"
    elif p == 0:
        return "🟢"
    elif p <= 5:          # adjust thresholds as needed
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change in % from previous month to current month. Increase = BAD (red), decrease = GOOD (green)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])


# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_harm_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())
    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')   # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in harm_indicators.items():
        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        indicator_row["Overall"] = fmt_cell(full_filtered_df[col])

        for m in months_to_display:
            df_m = full_filtered_df[full_filtered_df['month_period'] == m]
            indicator_row[m.strftime('%b %Y')] = fmt_cell(df_m[col])

        indicator_row["Status"] = status_icon(pct(full_filtered_df[col]))
        indicator_row["Trend"] = trend_icon(pct(df_trend_month[col]), pct(df_prev_month[col]))
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    ordered_columns = ["Indicator", "Overall"] + [m.strftime('%b %Y') for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_harm_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Partograph Review Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Clean column names
df.columns = df.columns.str.strip()

# Robust datetime parsing
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
partograph_indicators = {
    'partogragh_initiation': 'Partograph initiated at correct time',
    'partograph_half_hr': 'Contractions recorded every 30 minutes',
    'fetal_heart_correct': 'Foetal heart rate correctly recorded',
    'maternal_pulse_correct': 'Maternal pulse correctly recorded',
    'bp_4hrs': 'Blood pressure recorded every 4 hours',
    'partograph_summary_filled': 'Partograph completed after delivery',
    'delivery_method_filled': 'Delivery method correctly documented',
    'blood_loss_estimation': 'Estimated blood loss correctly recorded',
    'partograph_accuracy_concern': 'No data accuracy concerns'
}

# ----------------------------
# INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_score(series, col):

    clean = series[~series.isin(invalid_values)].dropna()
    clean = normalize(clean)
    clean_lower = clean.str.lower()

    if col == "partograph_accuracy_concern":
        no_safe = (clean_lower == "no").sum()
        yes_risk = (clean_lower == "yes").sum()

        denom = no_safe + yes_risk
        if denom == 0:
            return np.nan

        return (no_safe / denom) * 100

    yes = (clean_lower == "yes").sum()
    no = (clean_lower == "no").sum()

    denom = yes + no

    if denom == 0:
        return np.nan

    return (yes / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])

# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_partograph_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in partograph_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_score(df_trend_month[col], col)
        previous_score = compute_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_partograph_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))


In [ ]:
# @title Labor Care Guide Scorecard

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# ----------------------------
# DATA IMPORTATION
# ----------------------------
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Clean column names
df.columns = df.columns.str.strip()

# Ensure '_uuid' is available for unique delivery counting
if '_uuid' not in df.columns:
    df['_uuid'] = df.index

# Robust datetime parsing
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted'])  # Drop rows where date couldn't be parsed
df['month_period'] = df['date_submitted'].dt.to_period('M')

# ----------------------------
# POWER BI-STYLE DROPDOWN SLICER
# ----------------------------
class Slicer:
    """Dropdown header (shows 'All' / selection) that opens a panel with a search box,
    Select all and a scrollable checkbox list. Nothing ticked = All.
    Options can be rebuilt with set_options() (used for the County -> Facility cascade)."""

    BLANK = "(Blank)"

    def __init__(self, data, column, title, width='230px', height='200px'):
        self.column = column
        self._callbacks = []
        self._busy = False
        self._open = False
        self.options = []
        self.checks = {}

        # Title bar + dropdown header
        self.title = widgets.HTML(
            f"<div style='background:#2b8aa8; color:white; font-weight:bold; "
            f"padding:4px 8px; width:{width}; box-sizing:border-box;'>{title}</div>"
        )
        self.header = widgets.Button(
            description='All  ▾',
            layout=widgets.Layout(width=width, height='30px', margin='0')
        )

        # Panel contents
        self.search = widgets.Text(
            placeholder='🔍 Search',
            layout=widgets.Layout(width=width)
        )
        self.select_all = widgets.Checkbox(
            value=False, description='Select all', indent=False,
            layout=widgets.Layout(width='auto', margin='4px 0 0 0')
        )
        self.list_box = widgets.VBox(
            [],
            layout=widgets.Layout(
                height=height, width=width, overflow='auto',
                border='1px solid #ddd', padding='4px'
            )
        )
        self.panel = widgets.VBox(
            [self.search, self.select_all, self.list_box],
            layout=widgets.Layout(display='none', width=width, border='1px solid #ccc', padding='4px')
        )

        # Events
        self.header.on_click(self._toggle)
        self.search.observe(self._on_search, names='value')
        self.select_all.observe(self._on_select_all, names='value')

        self.widget = widgets.VBox(
            [self.title, self.header, self.panel],
            layout=widgets.Layout(margin='0 16px 0 0', align_items='flex-start')
        )
        self.widget.add_class('slicer-root')
        self.header.add_class('slicer-header')
        self.panel.add_class('slicer-panel')

        # Build the checkbox list from the data
        self.set_options(data[column])

    # ---- (re)build the option list ----
    def set_options(self, column_values):
        """Rebuild the list from a pandas Series. Ticks that are still valid are kept."""
        keep = {str(o) for o in self.options if o in self.checks and self.checks[o].value}

        values = sorted(column_values.dropna().unique().tolist())
        has_blank = column_values.isna().any()

        self._busy = True
        self.options = ([self.BLANK] if has_blank else []) + values
        self.checks = {
            opt: widgets.Checkbox(
                value=(str(opt) in keep), description=str(opt), indent=False,
                layout=widgets.Layout(width='auto', margin='0', flex='0 0 auto')
            )
            for opt in self.options
        }
        for chk in self.checks.values():
            chk.observe(self._on_check, names='value')

        self.search.value = ''
        self.list_box.children = list(self.checks.values())
        self._busy = False

        self._sync_select_all()
        self._update_header()

    # ---- helpers ----
    def _visible_options(self):
        term = self.search.value.strip().lower()
        return [o for o in self.options if term in str(o).lower()]

    def _sync_select_all(self):
        visible = self._visible_options()
        all_on = bool(visible) and all(self.checks[o].value for o in visible)
        self._busy = True
        self.select_all.value = all_on
        self._busy = False

    def _update_header(self):
        ticked = [str(o) for o in self.options if self.checks[o].value]
        if not ticked or len(ticked) == len(self.options):
            text = "All"
        elif len(ticked) <= 2:
            text = ", ".join(ticked)
        else:
            text = f"{len(ticked)} selected"
        if len(text) > 24:
            text = text[:22] + "…"
        arrow = "▴" if self._open else "▾"
        self.header.description = f"{text}  {arrow}"

    def _fire(self):
        for cb in self._callbacks:
            cb()

    # ---- event handlers ----
    def _toggle(self, _):
        self._open = not self._open
        self.panel.layout.display = 'flex' if self._open else 'none'
        self._update_header()

    def _on_search(self, change):
        visible = self._visible_options()
        self.list_box.children = [self.checks[o] for o in visible]
        self._sync_select_all()

    def _on_select_all(self, change):
        if self._busy:
            return
        self._busy = True
        for o in self._visible_options():          # only affects what the search shows
            self.checks[o].value = change['new']
        self._busy = False
        self._update_header()
        self._fire()

    def _on_check(self, change):
        if self._busy:
            return
        self._sync_select_all()
        self._update_header()
        self._fire()

    # ---- public API ----
    @property
    def selected(self):
        """List of ticked values (None stands for Blank). Empty list = no filter."""
        return [None if o == self.BLANK else o for o in self.options if self.checks[o].value]

    def on_change(self, callback):
        self._callbacks.append(callback)


def apply_slicer(data, column, selected):
    """Filter rows to the ticked values. Empty selection = All."""
    if not selected:
        return data
    mask = data[column].isin([v for v in selected if v is not None])
    if None in selected:
        mask = mask | data[column].isna()
    return data[mask]

# ----------------------------
# INDICATORS
# ----------------------------
lcg_indicators = {
    'lcg_initiated_correct': 'LCG initiated at correct time',
    'contractions_monitored_1st': 'Contractions recorded first stage',
    'fhr_monitored_1st': 'Foetal heart recorded first stage',
    'bp_monitored_recorded': 'Blood pressure correctly recorded',
    'contractions_monitored_2nd': 'Contractions recorded second stage',
    'fhr_monitored_2nd': 'Foetal heart recorded second stage',
    'delivery_time_documented': 'Delivery time correctly documented',
    'delivery_method_documented': 'Delivery method correctly documented',
    'ebl_documented_correctly': 'Blood loss correctly documented',
    'lcg_accuracy_concerns': 'No data accuracy concerns'
}

# ----------------------------
# INVALID VALUES
# ----------------------------
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]

# ----------------------------
# NORMALIZATION
# ----------------------------
def normalize(series):
    return series.astype(str).str.strip()

# ----------------------------
# SCORE FUNCTION
# ----------------------------
def compute_score(series, col):

    clean = series[~series.isin(invalid_values)].dropna()
    clean = normalize(clean)
    clean_lower = clean.str.lower()

    if col == "lcg_accuracy_concerns":
        no_safe = (clean_lower == "no").sum()
        yes_risk = (clean_lower == "yes").sum()

        denom = no_safe + yes_risk
        if denom == 0:
            return np.nan

        return (no_safe / denom) * 100

    yes = (clean_lower == "yes").sum()
    no = (clean_lower == "no").sum()

    denom = yes + no

    if denom == 0:
        return np.nan

    return (yes / denom) * 100

# ----------------------------
# STATUS + TREND
# ----------------------------
def status_icon(score):
    if pd.isna(score):
        return "⚪"
    elif score >= 80:
        return "🟢"
    elif score >= 60:
        return "🟠"
    else:
        return "🔴"

def trend_icon(current, previous):
    """Change from previous month to current month, shown as (#.#%)."""
    if pd.isna(current) or pd.isna(previous):
        return '<span style="color:grey; font-weight:bold;">▬ N/A</span>'

    diff = current - previous
    diff_str = f" ({abs(diff):.1f}%)"

    if diff > 0:
        return f'<span style="color:green; font-size:14px; font-weight:bold;">▲{diff_str}</span>'
    elif diff < 0:
        return f'<span style="color:red; font-size:14px; font-weight:bold;">▼{diff_str}</span>'
    else:
        return f'<span style="color:grey; font-size:14px; font-weight:bold;">▬{diff_str}</span>'

# ----------------------------
# FILTER WIDGETS
# ----------------------------
county_slicer = Slicer(df, 'county', 'County')
facility_slicer = Slicer(df, 'facility', 'Facility')

def update_facility_options(*_):
    """Show only facilities belonging to the selected county(ies). No county ticked = all."""
    subset = apply_slicer(df, 'county', county_slicer.selected)
    facility_slicer.set_options(subset['facility'])

# Month range (From / To). "All" = no limit.
available_months = sorted(df['month_period'].dropna().unique().tolist())
month_options = [("All", None)] + [(m.strftime('%b %Y'), m) for m in available_months]

start_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="From:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)
end_month_widget = widgets.Dropdown(
    options=month_options, value=None, description="To:",
    layout=widgets.Layout(width='220px'), style={'description_width': '45px'}
)

date_hint = widgets.HTML(
    f"<span style='color:grey; font-size:12px;'>Data available: "
    f"{available_months[0].strftime('%b %Y')} to {available_months[-1].strftime('%b %Y')}. "
    f"Month range: All = no limit. Nothing ticked in a list = All.</span>"
)

# ----------------------------
# SCORECARD FUNCTION
# ----------------------------
def build_lcg_scorecard(counties, facilities, start_month, end_month):

    # 1) Apply the categorical (slicer) filters. Nothing ticked = All.
    base_df = df.copy()
    base_df = apply_slicer(base_df, 'county', counties)
    base_df = apply_slicer(base_df, 'facility', facilities)

    if base_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # 2) Apply the month range (inclusive of both the From and To months).
    if start_month is not None and end_month is not None and start_month > end_month:
        display(HTML("<h3>The 'From' month must be on or before the 'To' month.</h3>"))
        return

    full_filtered_df = base_df.copy()
    if start_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] >= start_month]
    if end_month is not None:
        full_filtered_df = full_filtered_df[full_filtered_df['month_period'] <= end_month]

    if full_filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters and month range.</h3>"))
        return

    # Months shown as columns = months that fall inside the selected range
    months_to_display = sorted(full_filtered_df['month_period'].dropna().unique().tolist())

    if not months_to_display:
        display(HTML("<h3>No monthly data available for trend calculation.</h3>"))
        return

    # ---- TREND: 'To' month vs the month before it ----
    if end_month is not None:
        trend_month = end_month
    else:
        trend_month = pd.Period(pd.Timestamp.today(), freq='M')  # current month
    prev_month = trend_month - 1

    # Taken from base_df (not month-limited) so the previous month is
    # available even if it is outside the selected range.
    df_trend_month = base_df[base_df['month_period'] == trend_month]
    df_prev_month = base_df[base_df['month_period'] == prev_month]

    rows = []

    for col, label in lcg_indicators.items():

        if col not in df.columns:
            continue

        indicator_row = {"Indicator": label}

        # Score for the selected filters + month range
        overall_score = compute_score(full_filtered_df[col], col)
        indicator_row["Score"] = f"{overall_score:.1f}%" if not pd.isna(overall_score) else "N/A"

        # Scores for each month in the range
        for current_m_period in months_to_display:
            df_for_month = full_filtered_df[full_filtered_df['month_period'] == current_m_period]
            m_score = compute_score(df_for_month[col], col)
            indicator_row[f"{current_m_period.strftime('%b %Y')}"] = f"{m_score:.1f}%" if not pd.isna(m_score) else "N/A"

        # Trend: 'To' month vs the month before it
        current_score = compute_score(df_trend_month[col], col)
        previous_score = compute_score(df_prev_month[col], col)

        indicator_row["Status"] = status_icon(overall_score)
        indicator_row["Trend"] = trend_icon(current_score, previous_score)
        rows.append(indicator_row)

    if not rows:
        display(HTML("<h3>No indicator columns found for this section.</h3>"))
        return

    scorecard_df = pd.DataFrame(rows)

    # Reorder columns: Indicator, Score, monthly scores (chronological), Status, Trend
    ordered_columns = ["Indicator", "Score"] + [f"{m.strftime('%b %Y')}" for m in months_to_display] + ["Status", "Trend"]
    scorecard_df = scorecard_df[ordered_columns]

    start_label = start_month.strftime('%b %Y') if start_month is not None else 'start'
    end_label = end_month.strftime('%b %Y') if end_month is not None else 'latest'
    display(HTML(
        f"<p style='margin:4px 0;'><b>Month range:</b> {start_label} to {end_label} &nbsp;|&nbsp; "
        f"<b>Trend:</b> {trend_month.strftime('%b %Y')} vs {prev_month.strftime('%b %Y')}</p>"
    ))
    display(
        scorecard_df.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th',
             'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis='index')
    )

# ----------------------------
# RUN DASHBOARD
# ----------------------------
out = widgets.Output()

def refresh(*_):
    with out:
        clear_output(wait=True)
        build_lcg_scorecard(
            county_slicer.selected,
            facility_slicer.selected,
            start_month_widget.value,
            end_month_widget.value
        )

# Order matters: County change first narrows the Facility list, then the scorecard redraws.
county_slicer.on_change(update_facility_options)
county_slicer.on_change(refresh)
facility_slicer.on_change(refresh)
start_month_widget.observe(refresh, names='value')
end_month_widget.observe(refresh, names='value')

month_box = widgets.VBox([
    widgets.HTML(
        "<div style='background:#2b8aa8; color:white; font-weight:bold; "
        "padding:4px 8px; width:220px; box-sizing:border-box;'>Month range</div>"
    ),
    start_month_widget, end_month_widget
])
ui = widgets.VBox([
    widgets.HBox(
        [county_slicer.widget, facility_slicer.widget, month_box],
        layout=widgets.Layout(align_items='flex-start')
    ),
    date_hint
])

display(ui, out)
refresh()

# ----------------------------
# AUTO-CLOSE DROPDOWNS ON OUTSIDE CLICK
# ----------------------------
display(Javascript("""
(function () {
  if (window._slicerCleanup) { window._slicerCleanup(); }

  // Close every open slicer (header text ends in ▴), except the one containing 'keepTarget'
  function closeSlicers(keepTarget) {
    document.querySelectorAll('button').forEach(function (btn) {
      if (!/▴\\s*$/.test(btn.textContent)) return;
      var root = btn.closest('.widget-vbox') || btn.parentElement;
      if (keepTarget && root && root.contains(keepTarget)) return;
      btn.click();   // uses the existing toggle so the arrow and label update
    });
  }

  function onDown(e)  { closeSlicers(e.target); }
  function onBlur()   { closeSlicers(null); }
  function onLeave()  { closeSlicers(null); }

  document.addEventListener('mousedown', onDown, true);
  document.addEventListener('touchstart', onDown, true);
  window.addEventListener('blur', onBlur);
  document.documentElement.addEventListener('mouseleave', onLeave);

  window._slicerCleanup = function () {
    document.removeEventListener('mousedown', onDown, true);
    document.removeEventListener('touchstart', onDown, true);
    window.removeEventListener('blur', onBlur);
    document.documentElement.removeEventListener('mouseleave', onLeave);
  };
  console.log('slicer auto-close active');
})();
"""))

In [ ]:
#@title ➡️ QuIPS Deep Dive: Exploring Insights Beneath the Surface

In [ ]:
# @title Birth Preparedness v1
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down'] # Using a different name to avoid conflict if original harm_indicators is a dict

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Descriptive indicators for Birth Preparedness
indicators_to_plot_birth_preparedness = {
    'hand_hygiene': 'Hand Hygiene',
    'ppe_gloves': 'PPE Gloves',
    'ppe_mask': 'PPE Mask',
    'ppe_gown': 'PPE Gown',
    'ppe_boots': 'PPE Boots',
    'ppe_goggles': 'PPE Goggles',
    'ppe_cap': 'PPE Cap',
    'nnr_equipment_preparation': 'NNR Equipment Preparation',
    'uterotonic_preparation': 'Uterotonic Preparation'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_birth_preparedness_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator using descriptive labels
    calculated_scores_birth_preparedness = {}
    for indicator_col, label in indicators_to_plot_birth_preparedness.items():
        if indicator_col in filtered_df.columns:
            calculated_scores_birth_preparedness[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_birth_preparedness = pd.Series(calculated_scores_birth_preparedness).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_birth_preparedness.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_birth_preparedness.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Birth Preparedness Scores')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_birth_preparedness):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_birth_preparedness_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Birth Preparedness v2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down'] # Using a different name to avoid conflict if original harm_indicators is a dict

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else: # This 'else' covers standard 'Yes'/'No' columns, including the new 'full_protective_clothing_combined' series
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Descriptive indicators for Birth Preparedness - UPDATED
indicators_to_plot_birth_preparedness = {
    'hand_hygiene': 'Hand Hygiene',
    'nnr_equipment_preparation': 'NNR Equipment Preparation',
    'uterotonic_preparation': 'Uterotonic Preparation',
    'full_protective_clothing_combined': 'Full Protective Clothing' # New combined indicator
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart - UPDATED LOGIC
def plot_birth_preparedness_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Define individual PPE columns for combination (these are the source columns)
    ppe_columns_for_combination = ['ppe_gloves', 'ppe_mask', 'ppe_gown', 'ppe_boots', 'ppe_goggles', 'ppe_cap']
    full_protective_clothing_key = 'full_protective_clothing_combined' # The key used in indicators_to_plot_birth_preparedness

    calculated_scores_birth_preparedness = {}

    for indicator_col_key, indicator_label in indicators_to_plot_birth_preparedness.items():
        if indicator_col_key == full_protective_clothing_key:
            # Special handling for 'Full Protective Clothing'
            # Check if all component PPE columns are present in the filtered DataFrame
            if all(col in filtered_df.columns for col in ppe_columns_for_combination):
                temp_ppe_df = filtered_df[ppe_columns_for_combination].copy()

                # Normalize component PPE responses to 'Yes', 'No', or NaN
                # A direct comparison with 'Yes'/'No' (case-insensitive) is safer.
                for col in ppe_columns_for_combination:
                    temp_ppe_df[col] = temp_ppe_df[col].astype(str).str.strip().str.lower().map({'yes': 'Yes', 'no': 'No'}).fillna(np.nan)


                # Initialize the combined series with NaN (representing unknown/uncomputable)
                full_protective_clothing_combined_series = pd.Series(np.nan, index=filtered_df.index, dtype=object)

                # Condition for 'Yes': All component PPEs must be 'Yes' in that row, and none should be NaN
                all_yes_mask = (temp_ppe_df == 'Yes').all(axis=1)
                full_protective_clothing_combined_series.loc[all_yes_mask] = 'Yes'

                # Condition for 'No': If not 'Yes' (meaning not all were 'Yes') AND at least one was 'No', OR if some are valid and some are invalid, but not all 'Yes'.
                # More simply, if a row contains at least one 'No' and is not already 'Yes' (from the all_yes_mask)
                any_no_mask = (temp_ppe_df == 'No').any(axis=1)
                full_protective_clothing_combined_series.loc[any_no_mask & ~all_yes_mask] = 'No'

                # If a row contains only NaNs for all PPEs, it remains NaN in full_protective_clothing_combined_series,
                # which compute_score_unified will correctly ignore.
                # If a row has some 'Yes' and some NaNs, but no 'No', it would currently be NaN. This matches
                # the "all of them" requirement - if one is unknown, it's not "all Yes".

                calculated_scores_birth_preparedness[indicator_label] = compute_score_unified(full_protective_clothing_combined_series, indicator_col_key)
            else:
                print("Warning: One or more individual PPE columns missing. 'Full Protective Clothing' score cannot be computed.")
                calculated_scores_birth_preparedness[indicator_label] = np.nan # Assign NaN if not computable
        else:
            # Normal handling for other indicators that are direct columns
            if indicator_col_key in filtered_df.columns:
                calculated_scores_birth_preparedness[indicator_label] = compute_score_unified(filtered_df[indicator_col_key], indicator_col_key)
            else:
                print(f"Warning: Indicator '{indicator_col_key}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting, dropping any NaNs if a score couldn't be computed
    indicator_scores_birth_preparedness = pd.Series(calculated_scores_birth_preparedness).dropna().sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_birth_preparedness.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_birth_preparedness.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Birth Preparedness Scores')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels outside the top of each bar, black and bold
    for i, v in enumerate(indicator_scores_birth_preparedness):
        if not pd.isna(v):
            ax.text(i, v + 2, f"{v:.1f}%", ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_birth_preparedness_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Technical Quality Immediately Following Birth
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Define global invalid values and specific valid responses for blood loss
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Define harm indicators list (used in compute_score_unified)
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Technical Quality Immediately Following Birth (21 indicators) with descriptive labels
indicators_to_plot = {
    'ruleout_twin': 'Rule Out Twin',
    'uterotonic_administered': 'Uterotonic Administered',
    'timely_uterotonic_administration': 'Timely Uterotonic Administration',
    'drying_stimulation': 'Drying Stimulation',
    'discard_wet_towel': 'Discard Wet Towel',
    'skin_to_skin': 'Skin To Skin',
    'adequate_skin_to_skin_time': 'Adequate Skin To Skin Time',
    'timely_cord_clamp': 'Timely Cord Clamp',
    'sterilized_blade_clamp': 'Sterilized Blade Clamp',
    'newborn_kept_warm': 'Newborn Kept Warm',
    'controlled_cord_traction': 'Controlled Cord Traction',
    'placenta_examination': 'Placenta Examination',
    'uterine_tone_assessed': 'Uterine Tone Assessed',
    'laceration_check': 'Laceration Check',
    'method_blood_loss': 'Method Blood Loss',
    'proper_vdrape_use': 'Proper Vdrape Use',
    'maternal_bleeding_assessed': 'Maternal Bleeding Assessed',
    'administer_vitamin_k': 'Administer Vitamin K',
    'timely_vitamink_administration': 'Timely Vitamink Administration',
    'provider_initiate_breastfeeding': 'Provider Initiate Breastfeeding',
    'timely_breastfeeding_initiation': 'Timely Breastfeeding Initiation'
}

# Dropdown for score ranges
score_range_widget = widgets.Dropdown(
    options=["All", "Below 50%", "51% - 80%", "81% - 90%", "91% - 99%", "100%"],
    value="All",
    description="Score Range:"
)

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_indicator_scores(county, facility, month, score_range):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator using descriptive labels
    calculated_scores = {}
    for indicator_col, label in indicators_to_plot.items():
        if indicator_col in filtered_df.columns:
            calculated_scores[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores = pd.Series(calculated_scores, dtype=float).sort_values(ascending=False) * 100

    # Apply score range filter
    if score_range != "All":
        if score_range == "Below 50%":
            indicator_scores = indicator_scores[indicator_scores < 50]
        elif score_range == "51% - 80%":
            indicator_scores = indicator_scores[(indicator_scores >= 51) & (indicator_scores <= 80)]
        elif score_range == "81% - 90%":
            indicator_scores = indicator_scores[(indicator_scores >= 81) & (indicator_scores <= 90)]
        elif score_range == "91% - 99%":
            indicator_scores = indicator_scores[(indicator_scores >= 91) & (indicator_scores <= 99)]
        elif score_range == "100%":
            indicator_scores = indicator_scores[indicator_scores == 100]

    # Check if there's any data to plot after filtering
    if indicator_scores.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(14, 7))
    ax = indicator_scores.plot(kind='bar', color='purple')

    plt.title('Technical Quality Immediately Following Birth Indicators')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=60, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=8, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_indicator_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget,
    score_range=score_range_widget
)

In [ ]:
# @title Maternal Vitals & Documentation
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# Define global invalid values and specific valid responses for blood loss
invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Define harm indicators list (used in compute_score_unified)
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])).sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])).sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Maternal Vitals & Documentation (8 indicators) with descriptive labels
indicators_to_plot = {
    'take_mother_bp': 'Take Mother Bp',
    'timely_mother_bp': 'Timely Mother Bp',
    'take_mother_temperature': 'Take Mother Temperature',
    'timely_mother_temp': 'Timely Mother Temp',
    'take_mother_pulse': 'Take Mother Pulse',
    'timely_mother_pulse': 'Timely Mother Pulse',
    'apgar_min1': 'Apgar Min1',
    'apgar_min5': 'Apgar Min5'
}

# Dropdown for score ranges
score_range_widget = widgets.Dropdown(
    options=["All", "Below 50%", "51% - 80%", "81% - 90%", "91% - 99%", "100%"],
    value="All",
    description="Score Range:"
)

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_indicator_scores(county, facility, month, score_range):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator using descriptive labels
    calculated_scores = {}
    for indicator_col, label in indicators_to_plot.items():
        if indicator_col in filtered_df.columns:
            calculated_scores[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores = pd.Series(calculated_scores, dtype=float).sort_values(ascending=False) * 100

    # Apply score range filter
    if score_range != "All":
        if score_range == "Below 50%":
            indicator_scores = indicator_scores[indicator_scores < 50]
        elif score_range == "51% - 80%":
            indicator_scores = indicator_scores[(indicator_scores >= 51) & (indicator_scores <= 80)]
        elif score_range == "81% - 90%":
            indicator_scores = indicator_scores[(indicator_scores >= 81) & (indicator_scores <= 90)]
        elif score_range == "91% - 99%":
            indicator_scores = indicator_scores[(indicator_scores >= 91) & (indicator_scores <= 99)]
        elif score_range == "100%":
            indicator_scores = indicator_scores[indicator_scores == 100]

    # Check if there's any data to plot after filtering
    if indicator_scores.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(14, 7))
    ax = indicator_scores.plot(kind='bar', color='purple')

    plt.title('Maternal Vitals & Documentation Indicators')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=60, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=8, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_indicator_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget,
    score_range=score_range_widget
)

In [ ]:
# @title Infection Prevention
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function (for self-containment)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Infection Prevention with descriptive labels
indicators_to_plot_infection = {
    'sharps_disposal': 'Sharps Disposal',
    'waste_disposal': 'Waste Disposal'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_infection_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator using descriptive labels
    calculated_scores_infection = {}
    for indicator_col, label in indicators_to_plot_infection.items():
        if indicator_col in filtered_df.columns:
            calculated_scores_infection[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_infection = pd.Series(calculated_scores_infection).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_infection.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(10, 6))
    ax = indicator_scores_infection.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Infection Prevention Scores')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_infection):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_infection_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Respectful Maternity Care
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function (for self-containment)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Respectful Maternity Care
indicators_to_plot_rmc = {
    'birth_companion': 'Birth companion present',
    'provider_explanations': 'Provider explained process to mother clearly throughout',
    'spoken_to_directly': 'Provider spoke to the mother directly throughout',
    'spoken_to_kindly': 'Provider spoke to the mother kindly throughout',
    'rmc_no_disrespect_to_mother': 'Mother was treated with respect throughout'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_rmc_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_rmc = {}
    for indicator, label in indicators_to_plot_rmc.items():
        if indicator in filtered_df.columns:
            calculated_scores_rmc[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_rmc = pd.Series(calculated_scores_rmc).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_rmc.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_rmc.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Respectful Maternity Care Scores')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right', wrap=True)
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_rmc):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_rmc_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Avoidance of Harmful Practices
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function (for self-containment)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        # For harm indicators, 'No' means the harmful practice was avoided (good score)
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Avoidance of Harmful Practices, with descriptive labels
indicators_to_plot_harm = {
    'fundal_pressure': 'No instance of fundal pressure',
    'perineum_stretching': 'No instance of perineum stretching',
    'newborn_slapped': 'No instance of newborn slapping',
    'newborn_upside_down': 'No instance of newborn being held upside down'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_harm_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_harm = {}
    for indicator, label in indicators_to_plot_harm.items():
        if indicator in filtered_df.columns:
            # Use the unified score function, which handles harm indicators by counting 'No' as a positive
            calculated_scores_harm[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_harm = pd.Series(calculated_scores_harm).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_harm.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_harm.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Avoidance of Harmful Practices Scores')
    plt.ylabel('Proportion of "No" (Harm Avoided) (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_harm):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_harm_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Partograph Review
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function (for self-containment)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        # For these indicators, 'No' means the issue was avoided (good score)
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Partograph Review, with descriptive labels
indicators_to_plot_partograph = {
    'partogragh_initiation': 'Partograph initiated at correct time',
    'partograph_half_hr': 'Contractions recorded every 30 minutes',
    'fetal_heart_correct': 'Fetal heart rate correctly recorded',
    'maternal_pulse_correct': 'Maternal pulse correctly recorded',
    'bp_4hrs': 'Blood pressure recorded every 4 hours',
    'partograph_summary_filled': 'Partograph completed after delivery',
    'delivery_method_filled': 'Delivery method correctly documented',
    'blood_loss_estimation': 'Estimated blood loss correctly recorded',
    'partograph_accuracy_concern': 'No data accuracy concerns'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_partograph_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_partograph = {}
    for indicator, label in indicators_to_plot_partograph.items():
        if indicator in filtered_df.columns:
            # Use the unified score function, which handles 'No' as positive for 'partograph_accuracy_concern'
            calculated_scores_partograph[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_partograph = pd.Series(calculated_scores_partograph).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_partograph.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_partograph.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Partograph Review Scores')
    plt.ylabel('Proportion of Positive Response (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_partograph):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_partograph_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Labor Care Guide Review
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]
harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

# Unified score computation function (for self-containment)
def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns", "observer_intervention"]:
        # For these indicators, 'No' means the issue was avoided (good score)
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Labor Care Guide Review, with descriptive labels
indicators_to_plot_lcg = {
    'lcg_initiated_correct': 'LCG initiated at correct time',
    'contractions_monitored_1st': 'Contractions recorded first stage',
    'fhr_monitored_1st': 'Foetal heart recorded first stage',
    'bp_monitored_recorded': 'Blood pressure correctly recorded',
    'contractions_monitored_2nd': 'Contractions recorded second stage',
    'fhr_monitored_2nd': 'Foetal heart recorded second stage',
    'delivery_time_documented': 'Delivery time correctly documented',
    'delivery_method_documented': 'Delivery method correctly documented',
    'ebl_documented_correctly': 'Blood loss correctly documented',
    'lcg_accuracy_concerns': 'No data accuracy concerns',
    'observer_intervention': 'No observer intervention'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_lcg_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_lcg = {}
    for indicator, label in indicators_to_plot_lcg.items():
        if indicator in filtered_df.columns:
            # Use the unified score function, which handles 'No' as positive for relevant indicators
            calculated_scores_lcg[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_lcg = pd.Series(calculated_scores_lcg).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_lcg.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_lcg.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Labor Care Guide Review Scores')
    plt.ylabel('Proportion of Positive Response (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_lcg):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_lcg_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title ➡️ Select Multiple Deep Dive

In [ ]:
# @title Personal Protective Equipment (PPEs)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# New indicators requested by the user
indicators_to_plot_ppe = {
    'ppe_gloves': 'PPE Gloves',
    'ppe_mask': 'PPE Mask',
    'ppe_gown': 'PPE Gown',
    'ppe_boots': 'PPE Boots',
    'ppe_goggles': 'PPE Goggles',
    'ppe_cap': 'PPE Cap'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_ppe_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each new indicator
    calculated_scores_ppe = {}
    for indicator_col, label in indicators_to_plot_ppe.items():
        if indicator_col in filtered_df.columns:
            calculated_scores_ppe[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_ppe = pd.Series(calculated_scores_ppe).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_ppe.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_ppe.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Personal Protective Equipment Scores')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels inside the top of each bar
    for i, v in enumerate(indicator_scores_ppe):
        if not pd.isna(v):
            ax.text(i, v - 3, f"{v:.1f}%", ha='center', va='top',
                    fontsize=9, color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_ppe_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Explanations Indicators
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# New indicators for Explanations with descriptive labels
indicators_to_plot_explanations = {
    'explanations_labor_explained_mother': 'Labor Explained to Mother',
    'explanations_language_simple_clear': 'Language Simple and Clear',
    'explanations_procedure_explained_first': 'Procedure Explained First',
    'explanations_mother_informed_choices': 'Mother Informed of Choices',
    'explanations_consent_obtained_before': 'Consent Obtained Before',
    'explanations_progress_updated_regularly': 'Progress Updated Regularly',
    'explanations_questions_asked_mother': 'Questions Asked Mother'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_explanations_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator using descriptive labels
    calculated_scores_explanations = {}
    for indicator_col, label in indicators_to_plot_explanations.items():
        if indicator_col in filtered_df.columns:
            calculated_scores_explanations[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting, dropping any NaNs if a score couldn't be computed
    indicator_scores_explanations = pd.Series(calculated_scores_explanations).dropna().sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_explanations.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_explanations.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Explanations Indicators')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels outside the top of each bar, black and bold
    for i, v in enumerate(indicator_scores_explanations):
        if not pd.isna(v):
            ax.text(i, v + 2, f"{v:.1f}%", ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_explanations_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Provider Interaction Indicators
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

harm_indicators_list = ['fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down']

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list + ["partograph_accuracy_concern", "lcg_accuracy_concerns"]:
        yes_count = (clean == "No").sum()
        no_count = (clean == "Yes").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# New indicators for Provider Interaction with descriptive labels
indicators_to_plot_provider_interaction = {
    'directly_introduced_self_mother': 'Provider Introduced Self to Mother',
    'directly_addressed_mother_name': 'Provider Addressed Mother by Name',
    'directly_spoke_directly_mother': 'Provider Spoke Directly to Mother',
    'directly_encouraged_needs_expression': 'Provider Encouraged Needs Expression',
    'directly_engaged_companion_support': 'Provider Engaged Companion Support',
    'kindly_responded_mother_needs': 'Provider Kindly Responded to Mother Needs',
    'kindly_showed_empathy_compassion': 'Provider Showed Empathy/Compassion',
    'kindly_used_encouragement_praise': 'Provider Used Encouragement/Praise',
    'kindly_listened_actively_mother': 'Provider Listened Actively to Mother',
    'kindly_positive_attitude_respect': 'Provider Positive Attitude/Respect',
    'kindly_privacy_confidentiality_ensured': 'Provider Privacy/Confidentiality Ensured'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_provider_interaction_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator using descriptive labels
    calculated_scores_provider_interaction = {}
    for indicator_col, label in indicators_to_plot_provider_interaction.items():
        if indicator_col in filtered_df.columns:
            calculated_scores_provider_interaction[label] = compute_score_unified(filtered_df[indicator_col], indicator_col)
        else:
            print(f"Warning: Indicator '{indicator_col}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting, dropping any NaNs if a score couldn't be computed
    indicator_scores_provider_interaction = pd.Series(calculated_scores_provider_interaction).dropna().sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_provider_interaction.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_provider_interaction.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Provider Interaction Indicators')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels outside the top of each bar, black and bold
    for i, v in enumerate(indicator_scores_provider_interaction):
        if not pd.isna(v):
            ax.text(i, v + 2, f"{v:.1f}%", ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_provider_interaction_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Respectful Maternity Care: Disrespect and Abuse Indicators
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Updated harm_indicators_list to include new RMC D&A indicators for compute_score_unified
harm_indicators_list = [
    'fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down',
    'rmc_lack_of_privacy', 'rmc_non_consented_care', 'rmc_physical_abuse',
    'rmc_verbal_abuse', 'rmc_discrimination', 'rmc_abandonment',
    'rmc_denied_pain_relief', 'rmc_coerced_procedures', 'rmc_unable_to_observe'
]

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list:
        # Modified to count 'Yes' for these indicators, as requested
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Respectful Maternity Care: Disrespect and Abuse
indicators_to_plot_rmc_da = {
    'rmc_lack_of_privacy': 'Lack of Privacy',
    'rmc_non_consented_care': 'Non-Consented Care',
    'rmc_physical_abuse': 'Physical Abuse',
    'rmc_verbal_abuse': 'Verbal Abuse',
    'rmc_discrimination': 'Discrimination',
    'rmc_abandonment': 'Abandonment',
    'rmc_denied_pain_relief': 'Denied Pain Relief',
    'rmc_coerced_procedures': 'Coerced Procedures',
    'rmc_unable_to_observe': 'Unable to Observe'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_rmc_da_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_rmc_da = {}
    for indicator, label in indicators_to_plot_rmc_da.items():
        if indicator in filtered_df.columns:
            # For RMC D&A indicators, 'No' means the abuse was avoided (good score)
            calculated_scores_rmc_da[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_rmc_da = pd.Series(calculated_scores_rmc_da).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_rmc_da.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_rmc_da.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Respectful Maternity Care: Disrespect and Abuse Indicators')
    plt.ylabel('Proportion of "Yes" (%)') # Updated ylabel
    plt.xlabel('Indicator')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels outside the top of each bar, black and bold
    for i, v in enumerate(indicator_scores_rmc_da):
        if not pd.isna(v):
            ax.text(i, v + 2, f"{v:.1f}%", ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_rmc_da_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Areas of Concern in Partograph Review

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Updated harm_indicators_list to include new Partograph Accuracy Concern indicators
harm_indicators_list = [
    'fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down',
    'rmc_lack_of_privacy', 'rmc_non_consented_care', 'rmc_physical_abuse',
    'rmc_verbal_abuse', 'rmc_discrimination', 'rmc_abandonment',
    'rmc_denied_pain_relief', 'rmc_coerced_procedures', 'rmc_unable_to_observe',
    'accuracy_passenger', 'accuracy_passage', 'accuracy_powers',
    'accuracy_maternal_status_vital_signs', 'accuracy_labour_summary'
]

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list:
        # For these indicators, 'Yes' means a concern exists (we are counting 'Yes')
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Partograph Review: Accuracy Concerns
indicators_to_plot_partograph_accuracy = {
    'accuracy_passenger': 'Passenger',
    'accuracy_passage': 'Passage',
    'accuracy_powers': 'Powers',
    'accuracy_maternal_status_vital_signs': 'Maternal Vital Signs',
    'accuracy_labour_summary': 'Labour Summary'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_partograph_accuracy_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_partograph_accuracy = {}
    for indicator, label in indicators_to_plot_partograph_accuracy.items():
        if indicator in filtered_df.columns:
            calculated_scores_partograph_accuracy[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_partograph_accuracy = pd.Series(calculated_scores_partograph_accuracy).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_partograph_accuracy.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_partograph_accuracy.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Partograph Review: Accuracy Concerns')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Accuracy Concern Indicator')
    plt.xticks(rotation=45, ha='right', wrap=True)
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels outside the top of each bar, black and bold
    for i, v in enumerate(indicator_scores_partograph_accuracy):
        if not pd.isna(v):
            ax.text(i, v + 2, f"{v:.1f}%", ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_partograph_accuracy_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Labor Care Guide Review: Accuracy Concerns

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

# Data Importation
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

invalid_values = ["", "Unable to observe", "Not applicable", "NA", "N/A"]
blood_loss_valid = ["Measure with a calibrated jar", "V drape", "Weighing of bloodsoaked materials"]

# Updated harm_indicators_list to include new Partograph and LCG Accuracy Concern indicators
harm_indicators_list = [
    'fundal_pressure', 'perineum_stretching', 'newborn_slapped', 'newborn_upside_down',
    'rmc_lack_of_privacy', 'rmc_non_consented_care', 'rmc_physical_abuse',
    'rmc_verbal_abuse', 'rmc_discrimination', 'rmc_abandonment',
    'rmc_denied_pain_relief', 'rmc_coerced_procedures', 'rmc_unable_to_observe',
    'accuracy_passenger', 'accuracy_passage', 'accuracy_powers',
    'accuracy_maternal_status_vital_signs', 'accuracy_labour_summary',
    'lcg_accuracy_passenger', 'lcg_accuracy_passage', 'lcg_accuracy_powers',
    'lcg_accuracy_maternal_vital_signs', 'lcg_accuracy_labour_summary'
]

def compute_score_unified(series, col_name):
    clean = series[~series.isin(invalid_values)].dropna()

    yes_count = 0
    no_count = 0

    if col_name == "uterotonic_administered":
        valid_responses = series[~series.isin(invalid_values)]
        yes_count = (valid_responses != "None of the above").sum()
        no_count = (valid_responses == "None of the above").sum()
    elif col_name == "method_blood_loss":
        yes_count = clean.isin(blood_loss_valid).sum()
        no_count = len(clean) - yes_count
    elif col_name in ["provider_explanations", "spoken_to_directly", "spoken_to_kindly"]:
        yes_count = (clean.isin(["Yes", "Always"])) .sum()
        no_count = (clean.isin(["No", "Sometimes", "Rarely", "Never"])) .sum()
    elif col_name in harm_indicators_list:
        # For these indicators, 'Yes' means a concern exists (we are counting 'Yes')
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()
    else:
        yes_count = (clean == "Yes").sum()
        no_count = (clean == "No").sum()

    denom = yes_count + no_count

    if denom == 0:
        return np.nan

    return (yes_count / denom)


# Indicators for Labor Care Guide Review: Accuracy Concerns
indicators_to_plot_lcg_accuracy = {
    'lcg_accuracy_passenger': 'Passenger',
    'lcg_accuracy_passage': 'Passage',
    'lcg_accuracy_powers': 'Powers',
    'lcg_accuracy_maternal_vital_signs': 'Maternal Vital Signs',
    'lcg_accuracy_labour_summary': 'Labour Summary'
}

# Dropdowns for filters
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Function to create and display the bar chart
def plot_lcg_accuracy_scores(county, facility, month):
    filtered_df = df.copy()

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]

    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]

    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Calculate scores for each indicator
    calculated_scores_lcg_accuracy = {}
    for indicator, label in indicators_to_plot_lcg_accuracy.items():
        if indicator in filtered_df.columns:
            calculated_scores_lcg_accuracy[label] = compute_score_unified(filtered_df[indicator], indicator)
        else:
            print(f"Warning: Indicator '{indicator}' not found in DataFrame. Skipping.")

    # Convert to pandas Series for plotting
    indicator_scores_lcg_accuracy = pd.Series(calculated_scores_lcg_accuracy).sort_values(ascending=False) * 100

    # Check if there's any data to plot after filtering
    if indicator_scores_lcg_accuracy.empty:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, 'No data available for the selected filters.',
                 horizontalalignment='center', verticalalignment='center',
                 fontsize=14, color='gray')
        plt.axis('off')
        plt.show()
        return

    # Create the bar chart
    plt.figure(figsize=(12, 6))
    ax = indicator_scores_lcg_accuracy.plot(kind='bar', color='purple') # Set bar color to purple

    plt.title('Labor Care Guide Review: Accuracy Concerns')
    plt.ylabel('Proportion of "Yes" (%)')
    plt.xlabel('Accuracy Concern Indicator')
    plt.xticks(rotation=45, ha='right', wrap=True)
    plt.ylim(0, 100)

    # Remove grid lines
    plt.grid(False)

    # Customize borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('lightgrey')
    ax.spines['left'].set_color('lightgrey')

    # Add data labels outside the top of each bar, black and bold
    for i, v in enumerate(indicator_scores_lcg_accuracy):
        if not pd.isna(v):
            ax.text(i, v + 2, f"{v:.1f}%", ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Interactive dashboard
interact(
    plot_lcg_accuracy_scores,
    county=county_widget,
    facility=facility_widget,
    month=month_widget
)

In [ ]:
# @title Time-related Question Analysis (Descriptive Statistics)

In [ ]:
# @title

import pandas as pd
import numpy as np
import ipywidgets as widgets # Add import
from ipywidgets import interact # Add import
from IPython.display import display, HTML # Ensure HTML is imported for message

# Data Importation (for self-containment)
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRPEUzmFeJeSDdQySj9OinUnelYoHGaWtVaU0UIaKJU0v4UZoGp3VSTnlroBkl6GDmW1YZ9w1y4bAIY/pub?gid=1114469965&single=true&output=csv'
df = pd.read_csv(url)

# FIX: robust datetime parsing and month extraction to enable month filtering
df['date_submitted'] = pd.to_datetime(df['date_submitted'], errors='coerce')
df = df.dropna(subset=['date_submitted']) # Drop rows where date_submitted couldn't be parsed
df['month'] = df['date_submitted'].dt.strftime('%Y-%m') # Extract month in YYYY-MM format for consistency with other plots


# List of duration indicators requested by the user, with custom display labels
duration_indicators = {
    'uterotonic_administration_duration': 'Time to uterotonic administration',
    'skin_to_skin_duration': 'Skin to skin duration',
    'cord_clamp_duration': 'Delayed cord clamping',
    'mother_bp_duration' : 'Time to taking mother BP',
    'mother_temp_duration' : 'Time to taking mother temperature',
    'mother_pulse_duration' : 'Time to taking mother pulse',
    'vitamink_administration_duration' : 'Time to Vitamin K administration',
    'breastfeeding_initiation_duration' : 'Time to initiation of breastfeeding',
    'uterine_massage_duration' : 'Uterine Massage Duration',
    'oxytocin_administration_duration' : 'Time to oxytocin administration',
    'txa_administration_duration' : 'Time to TXA administration',
    'iv_establishment_duration' : 'Time to IV establishment'
}

# Helper function to format seconds into HH:MM:SS
def format_seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return 'N/A'
    # Ensure seconds are non-negative for formatting a duration
    if seconds < 0:
        # For display purposes, convert negative to positive and prefix with '-' if it happens
        return f"-{format_seconds_to_hhmmss(abs(seconds))}"

    td = pd.to_timedelta(seconds, unit='s')
    hours, remainder = divmod(td.total_seconds(), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}"

# Helper function to normalize indicator names
def normalize_indicator_name(name):
    return name.replace('_', ' ').title()

# Define filter widgets
county_widget = widgets.Dropdown(
    options=["All"] + sorted(df['county'].dropna().unique().tolist()),
    value="All",
    description="County:"
)

facility_widget = widgets.Dropdown(
    options=["All"] + sorted(df['facility'].dropna().unique().tolist()),
    value="All",
    description="Facility:"
)

month_widget = widgets.Dropdown(
    options=["All"] + sorted(df['month'].dropna().unique().tolist()),
    value="All",
    description="Month:"
)

# Main function to calculate and display statistics with filters
def display_duration_statistics(county, facility, month):
    filtered_df = df.copy() # Work on a copy

    if county != "All":
        filtered_df = filtered_df[filtered_df['county'] == county]
    if facility != "All":
        filtered_df = filtered_df[filtered_df['facility'] == facility]
    if month != "All":
        filtered_df = filtered_df[filtered_df['month'] == month]

    # Initialize a list to store the statistics for each indicator
    statistics_data = []

    # Check if filtered_df is empty after applying filters
    if filtered_df.empty:
        display(HTML("<h3>No data available for the selected filters.</h3>"))
        return

    # Calculate statistics for each indicator
    for col, display_label in duration_indicators.items():
        # Determine the indicator name for display, using custom label if provided, else normalizing
        indicator_name_for_display = display_label if display_label else normalize_indicator_name(col)

        if col in filtered_df.columns: # Use filtered_df here
            # Convert duration column from HH:MM:SS string to timedelta, then to total seconds
            series_seconds = pd.to_timedelta(filtered_df[col], errors='coerce').dt.total_seconds().dropna()

            # Filter out negative durations as they are not logically meaningful
            series_seconds = series_seconds[series_seconds >= 0]

            if not series_seconds.empty:
                mean_val = series_seconds.mean()
                median_val = series_seconds.median()
                max_val = series_seconds.max()
                min_val = series_seconds.min()
                range_val = max_val - min_val
                statistics_data.append({
                    'Indicator': indicator_name_for_display,
                    'Mean': format_seconds_to_hhmmss(mean_val),
                    'Median': format_seconds_to_hhmmss(median_val),
                    'Max': format_seconds_to_hhmmss(max_val),
                    'Min': format_seconds_to_hhmmss(min_val),
                    'Range': format_seconds_to_hhmmss(range_val)
                })
            else:
                statistics_data.append({
                    'Indicator': indicator_name_for_display,
                    'Mean': 'N/A (No valid positive durations)',
                    'Median': 'N/A (No valid positive durations)',
                    'Max': 'N/A (No valid positive durations)',
                    'Min': 'N/A (No valid positive durations)',
                    'Range': 'N/A (No valid positive durations)'
                })
        else:
            statistics_data.append({
                'Indicator': indicator_name_for_display,
                'Mean': 'N/A (Column not found)',
                'Median': 'N/A (Column not found)',
                'Max': 'N/A (Column not found)',
                'Min': 'N/A (Column not found)',
                'Range': 'N/A (Column not found)'
            })

    # Create a DataFrame from the collected statistics
    df_statistics = pd.DataFrame(statistics_data)

    # Display the DataFrame with enhanced styling
    display(
        df_statistics.style
        .set_properties(**{'text-align': 'left', 'border-collapse': 'collapse'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
        ])
        .hide(axis="index") # Hide the default pandas index
    )

# Interactive dashboard
ui = widgets.HBox([county_widget, facility_widget, month_widget])

out = widgets.interactive_output(
    display_duration_statistics,
    {
        "county": county_widget,
        "facility": facility_widget,
        "month": month_widget
    }
)

display(ui)

<div style="margin-top: 50px;"></div>
### Duration Statistics Table

In [ ]:
display(out)

In [ ]:
# ============================================================
# @title QuIPS-FQA

!pip -q install ipywidgets pandas

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# ============================================================
# 1. LOAD DATA
# ============================================================

CSV_URL = (
    "https://docs.google.com/spreadsheets/d/e/"
    "2PACX-1vTY8wfkWOpoLXQWiwuwyuonErXHlaaMjaNr2YqvZORTGhh-xpT9AsHwlT7H2EhLD5WQiRGiIUefUas8/"
    "pub?gid=1341559872&single=true&output=csv"
)

df = pd.read_csv(CSV_URL)
df.columns = df.columns.str.strip()


# ============================================================
# 2. VALIDATE REQUIRED COLUMNS
# ============================================================

required_cols = [
    "county",
    "facility_code",
    "facility",
    "theme_id",
    "theme_name",
    "insight_question",
    "quips_observations",
    "quips_scored_responses",
    "quips_practice_rate",
    "quips_practice_level",
    "fqa_readiness_level",
    "fqa_ready_count",
    "fqa_partial_count",
    "fqa_not_ready_count",
    "fqa_unknown_count",
    "insight_quadrant",
    "fqa_ready_enablers",
    "fqa_gap_enablers"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns in the source CSV: {missing_cols}"
    )


# ============================================================
# 3. CLEAN DATA
# ============================================================

text_cols = [
    "county",
    "facility_code",
    "facility",
    "theme_id",
    "theme_name",
    "insight_question",
    "quips_practice_level",
    "fqa_readiness_level",
    "insight_quadrant",
    "fqa_ready_enablers",
    "fqa_gap_enablers"
]

for col in text_cols:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# Convert numeric fields
numeric_cols = [
    "quips_observations",
    "quips_scored_responses",
    "fqa_ready_count",
    "fqa_partial_count",
    "fqa_not_ready_count",
    "fqa_unknown_count"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    ).fillna(0)


# Standardize practice rate
df["quips_practice_rate"] = (
    df["quips_practice_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)

df["quips_practice_rate"] = pd.to_numeric(
    df["quips_practice_rate"],
    errors="coerce"
)

df["Practice rate"] = df["quips_practice_rate"].apply(
    lambda x: f"{x:.0f}%" if pd.notna(x) else "—"
)


# Normalized values for filtering
df["practice_filter"] = (
    df["quips_practice_level"]
    .str.strip()
    .str.lower()
)

df["readiness_filter"] = (
    df["fqa_readiness_level"]
    .str.strip()
    .str.lower()
)


# Consistent sorting
df = df.sort_values(
    ["county", "facility", "theme_name"]
).reset_index(drop=True)


# ============================================================
# 4. CREATE FILTER WIDGETS
# ============================================================

county_options = (
    ["All counties"]
    + sorted(
        df.loc[df["county"] != "", "county"].unique()
    )
)

county_widget = widgets.Dropdown(
    options=county_options,
    value="All counties",
    description="County:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="320px")
)


facility_widget = widgets.Dropdown(
    options=["All facilities"],
    value="All facilities",
    description="Facility:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="440px")
)


practice_levels = sorted(
    df.loc[
        df["quips_practice_level"] != "",
        "quips_practice_level"
    ].unique()
)

practice_widget = widgets.Dropdown(
    options=["All practice levels"] + practice_levels,
    value="All practice levels",
    description="Practice:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="320px")
)


readiness_levels = sorted(
    df.loc[
        df["fqa_readiness_level"] != "",
        "fqa_readiness_level"
    ].unique()
)

readiness_widget = widgets.Dropdown(
    options=["All readiness levels"] + readiness_levels,
    value="All readiness levels",
    description="FQA readiness:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="340px")
)


output = widgets.Output()


# ============================================================
# 5. UPDATE FACILITY OPTIONS WHEN COUNTY CHANGES
# ============================================================

def update_facilities(*args):

    if county_widget.value == "All counties":
        available = df.copy()
    else:
        available = df[
            df["county"] == county_widget.value
        ]

    facilities = (
        available[["facility_code", "facility"]]
        .drop_duplicates()
        .sort_values("facility")
    )

    facility_options = ["All facilities"] + [
        (
            f"{row.facility} ({row.facility_code})",
            row.facility
        )
        for row in facilities.itertuples(index=False)
    ]

    current_facility = facility_widget.value

    facility_widget.options = facility_options

    valid_values = [
        option[1] if isinstance(option, tuple) else option
        for option in facility_options
    ]

    if current_facility in valid_values:
        facility_widget.value = current_facility
    else:
        facility_widget.value = "All facilities"


# ============================================================
# 6. RENDER SCORECARD MATRIX
# ============================================================

def render_scorecard(*args):

    with output:
        clear_output(wait=True)

        filtered = df.copy()

        # -----------------------------------------------
        # County filter
        # -----------------------------------------------
        if county_widget.value != "All counties":
            filtered = filtered[
                filtered["county"] == county_widget.value
            ]

        # -----------------------------------------------
        # Facility filter
        # -----------------------------------------------
        if facility_widget.value != "All facilities":
            filtered = filtered[
                filtered["facility"] == facility_widget.value
            ]

        # -----------------------------------------------
        # QuIPS practice-level filter
        # -----------------------------------------------
        if practice_widget.value != "All practice levels":
            filtered = filtered[
                filtered["practice_filter"]
                == practice_widget.value.strip().lower()
            ]

        # -----------------------------------------------
        # FQA readiness-level filter
        # -----------------------------------------------
        if readiness_widget.value != "All readiness levels":
            filtered = filtered[
                filtered["readiness_filter"]
                == readiness_widget.value.strip().lower()
            ]

        # -----------------------------------------------
        # Empty-state message
        # -----------------------------------------------
        if filtered.empty:
            display(HTML("""
                <div style="
                    padding:20px;
                    background:#F8F5FA;
                    border-left:5px solid #7F3F98;
                    color:#4B2C5E;
                    font-family:Arial,sans-serif;
                ">
                    <b>No matching records</b><br>
                    Adjust the filters to view the scorecard.
                </div>
            """))
            return

        # -----------------------------------------------
        # Determine displayed rows
        # -----------------------------------------------
        if facility_widget.value == "All facilities":
            index_cols = [
                "county",
                "facility",
                "theme_name"
            ]
        else:
            index_cols = ["theme_name"]

        display_cols = index_cols + [
            "insight_question",
            "quips_observations",
            "quips_scored_responses",
            "Practice rate",
            "quips_practice_level",
            "fqa_readiness_level",
            "fqa_ready_count",
            "fqa_partial_count",
            "fqa_not_ready_count",
            "fqa_unknown_count",
            "insight_quadrant",
            "fqa_ready_enablers",
            "fqa_gap_enablers"
        ]

        display_df = filtered[display_cols].copy()

        # -----------------------------------------------
        # Rename columns for display
        # -----------------------------------------------
        display_df = display_df.rename(columns={
            "county": "County",
            "facility": "Facility",
            "theme_name": "Insight theme",
            "insight_question": "Insight question",
            "quips_observations": "Observations",
            "quips_scored_responses": "Scored responses",
            "quips_practice_level": "Practice level",
            "fqa_readiness_level": "FQA readiness",
            "fqa_ready_count": "Ready",
            "fqa_partial_count": "Partial",
            "fqa_not_ready_count": "Not ready",
            "fqa_unknown_count": "Unknown",
            "insight_quadrant": "Insight quadrant",
            "fqa_ready_enablers": "FQA readiness enablers",
            "fqa_gap_enablers": "FQA gap enablers"
        })

        # -----------------------------------------------
        # Format readiness and gap enablers
        # -----------------------------------------------
        for col in [
            "FQA readiness enablers",
            "FQA gap enablers"
        ]:
            display_df[col] = (
                display_df[col]
                .str.replace("<br>", " • ", regex=False)
                .str.replace("<br/>", " • ", regex=False)
                .str.replace("<br />", " • ", regex=False)
                .str.replace("\n", " • ", regex=False)
                .str.strip()
            )

        # -----------------------------------------------
        # Cell colour rules
        # -----------------------------------------------
        def practice_colour(value):

            value = str(value).strip().lower()

            if value == "strong":
                return (
                    "background-color:#D9EAD3;"
                    "color:#274E13;font-weight:600"
                )

            elif value == "mixed":
                return (
                    "background-color:#FFF2CC;"
                    "color:#7F6000;font-weight:600"
                )

            elif value == "weak":
                return (
                    "background-color:#F4CCCC;"
                    "color:#990000;font-weight:600"
                )

            return ""


        def readiness_colour(value):

            value = str(value).strip().lower()

            if value == "ready":
                return (
                    "background-color:#D9EAD3;"
                    "color:#274E13;font-weight:600"
                )

            elif value in ["partial", "partially ready"]:
                return (
                    "background-color:#FFF2CC;"
                    "color:#7F6000;font-weight:600"
                )

            elif value in ["not ready", "not_ready"]:
                return (
                    "background-color:#F4CCCC;"
                    "color:#990000;font-weight:600"
                )

            return ""


        def quadrant_colour(value):

            value = str(value).strip().lower()

            if "enabled & practiced" in value:
                return (
                    "background-color:#D9EAD3;"
                    "color:#274E13"
                )

            elif "practice gap" in value:
                return (
                    "background-color:#F4CCCC;"
                    "color:#990000"
                )

            elif "mixed" in value:
                return (
                    "background-color:#FFF2CC;"
                    "color:#7F6000"
                )

            return ""


        # -----------------------------------------------
        # Build styled matrix
        # -----------------------------------------------
        styled = (
            display_df.style
            .map(
                practice_colour,
                subset=["Practice level"]
            )
            .map(
                readiness_colour,
                subset=["FQA readiness"]
            )
            .map(
                quadrant_colour,
                subset=["Insight quadrant"]
            )
            .set_properties(**{
                "text-align": "left",
                "vertical-align": "top",
                "white-space": "normal",
                "padding": "10px",
                "border-bottom": "1px solid #E5E7EB"
            })
            .set_properties(
                subset=[
                    "Observations",
                    "Scored responses",
                    "Practice rate",
                    "Ready",
                    "Partial",
                    "Not ready",
                    "Unknown"
                ],
                **{"text-align": "center"}
            )
            .set_table_styles([
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#4B2C5E"),
                        ("color", "white"),
                        ("font-weight", "600"),
                        ("text-align", "left"),
                        ("padding", "10px"),
                        ("position", "sticky"),
                        ("top", "0"),
                        ("z-index", "1")
                    ]
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                        ("font-family", "Arial,sans-serif"),
                        ("font-size", "12px")
                    ]
                },
                {
                    "selector": "caption",
                    "props": [
                        ("caption-side", "top"),
                        ("font-size", "18px"),
                        ("font-weight", "600"),
                        ("color", "#4B2C5E"),
                        ("text-align", "left"),
                        ("padding", "12px 0")
                    ]
                }
            ])
            .set_caption(
                "QuIPS × FQA Facility Insight Scorecard"
            )
            .hide(axis="index")
        )

        # -----------------------------------------------
        # Display matrix
        # -----------------------------------------------
        display(HTML("""
            <style>
                .scorecard-wrap {
                    width: 100%;
                    overflow-x: auto;
                    margin-top: 12px;
                }

                .scorecard-wrap table {
                    min-width: 1600px;
                }
            </style>
        """))

        display(HTML('<div class="scorecard-wrap">'))
        display(styled)
        display(HTML("</div>"))


# ============================================================
# 7. CONNECT WIDGETS
# ============================================================

county_widget.observe(
    update_facilities,
    names="value"
)

county_widget.observe(
    render_scorecard,
    names="value"
)

facility_widget.observe(
    render_scorecard,
    names="value"
)

practice_widget.observe(
    render_scorecard,
    names="value"
)

readiness_widget.observe(
    render_scorecard,
    names="value"
)


# ============================================================
# 8. DISPLAY DASHBOARD
# ============================================================

display(HTML("""
    <div style="
        font-family:Arial,sans-serif;
        color:#4B2C5E;
        margin-bottom:12px;
    ">
        <h2 style="margin-bottom:4px;">
            QuIPS × FQA Insight Scorecard
        </h2>

        <div style="color:#666;font-size:13px;">
            Filter by county, facility, observed practice,
            and facility readiness.
        </div>
    </div>
"""))

display(
    widgets.VBox([
        widgets.HBox(
            [county_widget, facility_widget],
            layout=widgets.Layout(
                gap="12px",
                flex_flow="row wrap",
                align_items="center"
            )
        ),
        widgets.HBox(
            [practice_widget, readiness_widget],
            layout=widgets.Layout(
                gap="12px",
                flex_flow="row wrap",
                align_items="center"
            )
        )
    ], layout=widgets.Layout(gap="12px"))
)

display(output)


# ============================================================
# 9. INITIALIZE
# ============================================================

update_facilities()
render_scorecard()